<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;background:linear-gradient(120deg,#14213d 0%,#2a4a9b 45%,#4285f4 78%,#c96442 100%);padding:30px 32px;border-radius:14px;color:#ffffff;"><div style="font-size:12.5px;letter-spacing:2.5px;text-transform:uppercase;color:#ffffff;">Google Quantum AI · Verifiable Quantum Circuit Compilation</div><div style="font-size:34px;font-weight:800;line-height:1.15;margin-top:8px;color:#ffffff;">How a quantum-circuit score is built, how we got to 33, and where it can still move</div><div style="font-size:17px;line-height:1.5;margin-top:10px;color:#ffffff;">A guided tour of the whole system: what the score measures, what each public step was worth, an ablation of the ideas, the floors of every lever, the best place to invest next, and what a human can and cannot follow in circuits found by AI search. Includes everything we tried that failed, with numbers.</div><div style="margin-top:14px;"><span style="display:inline-block;background:#ffffff;color:#14213d;border-radius:12px;padding:3px 12px;font-size:13px;font-weight:700;margin:4px 6px 0 0;">public best 33.10186</span><span style="display:inline-block;background:#ffffff;color:#14213d;border-radius:12px;padding:3px 12px;font-size:13px;font-weight:700;margin:4px 6px 0 0;">all circuits are others’ public work</span><span style="display:inline-block;background:#ffffff;color:#14213d;border-radius:12px;padding:3px 12px;font-size:13px;font-weight:700;margin:4px 6px 0 0;">runs in seconds, no input</span><span style="display:inline-block;background:#ffffff;color:#14213d;border-radius:12px;padding:3px 12px;font-size:13px;font-weight:700;margin:4px 6px 0 0;">Apache-2.0</span></div></div>

In [ ]:
IMAGE_SLOTS = False   # True shows an empty dashed slot for any picture that is missing
IMAGES = {'hero': '...', 'staircase': '...', 'ladder': '...', 'map': '...', 'doors': '...'}   # name -> base64 JPEG (pictures made with ChatGPT for this notebook)


<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.6;color:#111111;background:#eaf6f0;border-left:6px solid #0f9a68;border-radius:8px;padding:12px 18px;margin:10px 0;"><b>What you will learn.</b><ol style="margin:6px 0 4px 0;padding-left:22px;color:#111111;"><li><b>The score is a product of three cost ratios</b>, and each task has its own cost formula. One page of arithmetic tells you what any idea is worth before you build it.</li><li><b>How the public score climbed from 24.65 to 33.10</b>, and which steps were one big jump by one team and which were many small polishes.</li><li><b>An ablation</b>: which part of the score each task and each documented idea is responsible for.</li><li><b>The floors</b>: how low each lever can go, what we proved cannot be removed, and what is unknown.</li><li><b>Where to invest</b> next, in effort per point, and where the noise is bigger than the gain.</li><li><b>What a human can follow</b> in these circuits and what was found by search, plus a ladder of exercises if you want to take part.</li></ol></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;color:#111111;background:#ffffff;"><b>How to read this notebook.</b> Charts and short explanations come first; every code cell is collapsed (click to open). All numbers are written into the code as literals, so it runs in a few seconds with no input. Contents: <a href="#p1">1 the game</a> · <a href="#p2">2 the system map</a> · <a href="#p3">3 how we got here</a> · <a href="#p4">4 ablation</a> · <a href="#p5">5 floors</a> · <a href="#p6">6 where to invest</a> · <a href="#p7">7 human vs search</a> · <a href="#p8">8 what we tried and failed</a> · <a href="#credits">credits</a>.</div>

In [ ]:
def slot(name, caption):
    from IPython.display import HTML, display
    if name in IMAGES:
        display(HTML(f'<div style="text-align:center;background:#14213d;border-radius:12px;overflow:hidden;margin:8px 0;"><img src="data:image/jpeg;base64,{IMAGES[name]}" style="width:100%;display:block;"><div style="font-size:13px;color:#ffffff;background:#14213d;padding:6px 10px;">{caption}</div></div>'))
    elif IMAGE_SLOTS:
        display(HTML(f'<div style="border:2px dashed #8a94a6;border-radius:10px;padding:22px;text-align:center;color:#333333;background:#f6f7f9;font-size:14px;">picture slot: <b>{name}</b><br>{caption}</div>'))


In [ ]:
slot("hero", "Three machines, one for each task, each turning a tangled circuit into one number")


In [ ]:
# Plotting style and all measured numbers (literals: this notebook needs no input and runs in seconds).
import time, math
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
T0 = time.time()
INK, INK2, MUTED, RULE, SURF, NAVY = "#111111", "#333333", "#6b6b6b", "#d9d9d9", "#ffffff", "#0b1f4b"
C = ["#1f6fd1", "#d9541e", "#0f9a68", "#c98a00", "#d4558e", "#007a00", "#5843c9", "#d1302f"]
plt.rcParams.update({"font.family": "DejaVu Sans", "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
                     "axes.edgecolor": "#888888", "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
                     "axes.grid": True, "grid.color": "#ececec", "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 11.5, "axes.titlesize": 12.5, "axes.titleweight": "bold", "axes.titlecolor": INK, "axes.labelsize": 11.5,
                     "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 10.5, "lines.linewidth": 2, "legend.frameon": False,
                     "axes.axisbelow": True, "figure.dpi": 110})

def headline(fig, title, sub=None):
    """Figure title that states the takeaway (left aligned), with an optional dark-grey subtitle."""
    fig.text(0.012, 0.988, title, ha="left", va="top", fontsize=15.5, fontweight="bold", color=INK)
    if sub: fig.text(0.012, 0.988 - 0.36 / fig.get_figheight(), sub, ha="left", va="top", fontsize=11, color=INK2)
    top = 1 - ((0.54 + 0.2 * (sub.count("\n") + 1)) if sub else 0.46) / fig.get_figheight()
    fig.tight_layout(rect=(0, 0, 1, top))

SHOTS = 102_400                      # shots of one official DQI verifier run
# Official DQI verifier runs. cost = qubits x mean Toffolis x repetition factor (the factor grows with failed shots).
RUNS = pd.DataFrame([
    # candidate,                      base file,           Toffolis removed, salt,   value errors, repetition factor, cost,     base cost
    ("base 890-qubit line",           "890 q",              0,  "default",     0, None,   345.11e9, 345.11e9),
    ("110th inverse site (HW-31 leaf)", "890 q",            1,  "default",     2, 1.2014, 348.04e9, 345.11e9),
    ("base ThemBones 901-qubit file", "901 q",              0,  "default",     0, None,   3.327e11, 3.327e11),
    ("cascade96",                     "901 q",             96,  "default",    12, 1.2347, 3.447e11, 3.327e11),
    ("cascade96",                     "901 q",             96,  "alt-salt-1", 18, 1.2500, 3.490e11, 3.327e11),
    ("void2",                         "901 q",              2,  "default",    20, None,   3.504e11, 3.327e11),
    ("void2",                         "901 q",              2,  "alt-salt-1", 14, None,   3.463e11, 3.327e11),
], columns=["candidate", "base", "Toffolis removed", "salt", "value errors", "repetition factor", "cost", "base cost"])
RUNS["cost change %"] = (RUNS["cost"] / RUNS["base cost"] - 1) * 100
print("setup done")
# ---- constants of the three tasks and the public file sets (authors' published costs; see the credits)
REF = {"invsqrt": 2076774.233715282, "dqi": 24439463583672.93, "chem": 3355599588.3155365}
def gm(costs): return float(np.prod([REF[k] / costs[k] for k in REF]) ** (1 / 3))
SETS = {  # task costs; 'derived' = solved from the published board score of the set
    "28.15 level (17 Nov)":   {"invsqrt": 74797.0,  "dqi": 3.46410e11, "chem": 2.946423e8},
    "ThemBones 31.68 (20 Nov)": {"invsqrt": 63055.04, "dqi": 3.32669e11, "chem": None},
    "best public files (21 Nov)": {"invsqrt": 60955.33, "dqi": 3.050416e11, "chem": 252534924.82},
}
BOARD = {"28.15 level (17 Nov)": 28.15096, "ThemBones 31.68 (20 Nov)": 31.67924, "best public files (21 Nov)": 33.10186}
for k, s in SETS.items():          # fill the one unknown cost of each set from its board score (labelled 'derived')
    miss = [t for t in REF if s[t] is None]
    if miss:
        t = miss[0]; known = np.prod([REF[u] / s[u] for u in REF if u != t])
        s[t] = REF[t] / (BOARD[k] ** 3 / known)
print({k: {t: f"{v:,.4g}" for t, v in s.items()} for k, s in SETS.items()})
print({k: round(gm(s), 3) for k, s in SETS.items()}, "(board:", BOARD, ")")


<a id="p1"></a>
<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;background:linear-gradient(120deg,#14213d,#2a6fdb);color:#ffffff;padding:18px 24px;border-radius:12px;margin:22px 0 8px 0;"><div style="font-size:12.5px;letter-spacing:2px;text-transform:uppercase;color:#ffffff;">Part 1</div><div style="font-size:26px;font-weight:800;line-height:1.2;margin-top:4px;color:#ffffff;">The game: three machines, one score</div><div style="font-size:15.5px;margin-top:6px;color:#ffffff;">Why a Toffoli gate is the currency and why a qubit can be worth forty of them</div></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.65;color:#111111;background:#ffffff;padding:2px 4px;">The competition asks for three quantum circuits that must give the right answer when a verifier runs them: <b>invsqrt</b> (a phase that depends on 1/&radic;x for a 22-bit number x), <b>DQI</b> (a decoder circuit of a quantum optimisation method, 901 qubits in the current files) and <b>chemistry</b> (an energy-estimation circuit whose rotation angles are read from tables by small "loader" circuits, 405 qubits). Each circuit gets a <b>cost</b>; the cost is compared with a reference cost, and the score is the <b>geometric mean of the three ratios reference / cost</b>. A cheaper circuit is a higher ratio. Everything below follows from that one sentence.</div>

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(7, 3.3))
cards = [("invsqrt", "cost = qubits x (Toffolis + T/4)", "39 qubits x 1,616.8\n= 63,055", C[0]),
         ("DQI", "cost = R x qubits^2 x Toffolis", "1.191 x 901^2 x 343,998\n= 3.33e11", C[1]),
         ("chemistry", "cost = lambda x T x qubits", "46.93 x 13,434 x 405\n= 2.55e8", C[2])]
for ax, (n, f, ex, col) in zip(axs, cards):
    ax.axis("off"); ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes, fc="#f6f8fc", ec=col, lw=2.2))
    ax.text(0.5, 0.86, n, ha="center", va="center", fontsize=14, fontweight="bold", color=col, transform=ax.transAxes)
    ax.text(0.5, 0.58, f.replace("cost = ", "cost =\n"), ha="center", va="center", fontsize=9.6, color=INK, transform=ax.transAxes)
    ax.text(0.5, 0.18, ex, ha="center", va="center", fontsize=8.6, color=INK2, transform=ax.transAxes, wrap=True)
headline(fig, "Three cost recipes, one score: the product of three ratios", "Examples use the ThemBones 31.68 files (the DQI factor R is the extra cost of failed verifier shots).")
plt.show()


<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.55;max-width:100%;overflow-x:auto;margin:6px 0 14px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:2px;">About the figure: the three recipes</div><table style="border-collapse:collapse;color:#111111;background:#ffffff;"><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">How to read it</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Each box is one task. The line under the name is its cost formula; the last line plugs in the numbers of the ThemBones files.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">What it shows</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">invsqrt multiplies qubits by work; DQI multiplies the square of the qubits by its Toffolis and by R (the penalty for failed verifier shots); chemistry multiplies three factors. In all three the cost is a product, so a small cut in any factor cuts the whole cost.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">So what</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">The formulas decide what is cheap to improve. A qubit enters invsqrt once but DQI twice (squared), and R shows that a wrong shot costs money. Everything else in this notebook is these three lines applied.</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.6;color:#111111;background:#f3f6fc;border-left:6px solid #2a6fdb;border-radius:8px;padding:12px 18px;margin:10px 0;"><b>Why Toffoli?</b> A quantum computer can do many gates cheaply; the AND gate on three qubits (Toffoli) and the T rotation are the expensive ones, so the cost formulas count them. A T gate is priced as a quarter of a Toffoli, so invsqrt adds T/4 to its Toffoli count. <b>Why qubits matter so much:</b> the invsqrt cost is <i>qubits &times; work</i>, so with 39 qubits one qubit is 1/39 = 2.6% of the cost, and the DQI cost grows with the <i>square</i> of the qubits. One invsqrt qubit is worth about 42 invsqrt Toffolis (Part 6), and one DQI qubit about 750 DQI Toffolis.</div>

<a id="p2"></a>
<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;background:linear-gradient(120deg,#2a9d8f,#1e7166);color:#ffffff;padding:18px 24px;border-radius:12px;margin:22px 0 8px 0;"><div style="font-size:12.5px;letter-spacing:2px;text-transform:uppercase;color:#ffffff;">Part 2</div><div style="font-size:26px;font-weight:800;line-height:1.2;margin-top:4px;color:#ffffff;">The system map: what goes where, and who judges what</div><div style="font-size:15.5px;margin-top:6px;color:#ffffff;">One general map, then one zoom for each task, with every step in plain words</div></div>

In [ ]:
slot("map", "The whole system as a small city: one gate, three factories, inspectors on the roads, one scoreboard")


<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.65;color:#111111;background:#ffffff;padding:2px 4px;">The score looks like one number, but behind it is a pipeline. You hand in <b>one zip with 18 files</b>; each file is a quantum circuit (or a table of numbers) in a binary format called <i>kickmix</i>. A <b>verifier</b> built by the organisers runs every circuit and checks that it gives the right answer. Only circuits that pass are given a <b>cost</b>, and the three costs become the score. The map below shows the whole path; the three maps after it open up one task each.</div>

In [ ]:

def bx(ax, x, y, w, h, t, fc, ec=None, fs=8.4, tc="#111111", lw=1.4, bold=False):
    ax.add_patch(plt.Rectangle((x, y), w, h, fc=fc, ec=ec or fc, lw=lw, zorder=2))
    ax.text(x + w / 2, y + h / 2, t, ha="center", va="center", fontsize=fs, color=tc, zorder=3, fontweight="bold" if bold else "normal", linespacing=1.25)
def ar(ax, x1, y1, x2, y2, c="#555555"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", color=c, lw=1.5), zorder=1)

fig, ax = plt.subplots(figsize=(7, 5.3)); ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis("off"); ax.grid(False)
bx(ax, 1, 36, 15, 28, "submission.zip\n18 files\n(kickmix circuits\n+ one table)", "#e9ecf2", "#14213d", fs=8.2, bold=True)
rows = [(74, "invsqrt", C[0], "invsqrt.kmb\n1 file, 37-39 wires", "runs EVERY input\n1 to 3,133,452\n(200M random shots)\nerror <= tolerance", "qubits x\n(Toffolis + T/4)"),
        (43, "DQI", C[1], "dqi.kmb\n1 file, 901 wires\n(0.6 GB unpacked)", "102,400 random shots\nseed = hash of file\n0 value / phase errors", "R x qubits^2 x\nToffolis"),
        (12, "chemistry", C[2], "submission.npz +\n15 angle_k loaders\n405 wires", "error_1, error_2 <= 20 mHa\nenergy gap >= floor\nreads lambda", "lambda x T x\nqubits")]
for y, n, col, f, j, c in rows:
    bx(ax, 22, y, 20, 19, f, "#ffffff", col, fs=7.8); bx(ax, 49, y, 24, 19, j, "#f7f8fb", col, fs=7.6); bx(ax, 79, y, 20, 19, c, "#ffffff", col, fs=7.8)
    ax.text(22, y + 21.3, n, fontsize=9.5, fontweight="bold", color=col)
    ar(ax, 16, 50, 22, y + 9.5); ar(ax, 42, y + 9.5, 49, y + 9.5); ar(ax, 73, y + 9.5, 79, y + 9.5)
ax.text(49, 98, "judge (official verifier)", fontsize=8.5, color="#333333", style="italic"); ax.text(79, 98, "cost", fontsize=8.5, color="#333333", style="italic"); ax.text(22, 98, "files", fontsize=8.5, color="#333333", style="italic")
bx(ax, 22, 0, 77, 7, "score = geometric mean of the three ratios REF/cost", "#14213d", fs=8.2, tc="#ffffff", bold=True)
headline(fig, "From 18 files to one score: three judges, three costs, one geometric mean", "Cost formulas of Part 1; kickmix is the circuit format of the competition.")
plt.show()


<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.6;color:#111111;background:#eaf6f4;border-left:6px solid #2a9d8f;border-radius:8px;padding:12px 18px;margin:10px 0;"><b>Words you need (plain meaning).</b><table style="border-collapse:collapse;margin-top:6px;color:#111111;background:#ffffff;"><tr><td style="padding:5px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;vertical-align:top;">circuit</td><td style="padding:5px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">A list of gate steps acting on wires (qubits). Here it is a file; the organisers' tools can run it.</td></tr><tr><td style="padding:5px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;vertical-align:top;">wire / qubit</td><td style="padding:5px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">One quantum memory cell. More wires cost more, because every step is paid for all of them.</td></tr><tr><td style="padding:5px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;vertical-align:top;">Toffoli gate</td><td style="padding:5px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">A reversible AND of two wires into a third. The expensive gate: the cost counts them.</td></tr><tr><td style="padding:5px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;vertical-align:top;">T gate / rotation</td><td style="padding:5px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">A small turn of a wire, priced as a quarter of a Toffoli.</td></tr><tr><td style="padding:5px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;vertical-align:top;">measure and repair</td><td style="padding:5px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Measuring a temporary wire erases it for free; a cheap phase gate repairs the side effect (Part 7).</td></tr><tr><td style="padding:5px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;vertical-align:top;">shot</td><td style="padding:5px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">One random run of a circuit. Measurements give random outcomes, so a verifier runs many shots.</td></tr><tr><td style="padding:5px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;vertical-align:top;">verifier</td><td style="padding:5px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">The organisers' program that runs a circuit on test inputs and checks the answer against the exact value.</td></tr><tr><td style="padding:5px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;vertical-align:top;">salt / seed</td><td style="padding:5px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">A number that decides which random test shots the verifier draws. For DQI it is made from a hash of the file.</td></tr><tr><td style="padding:5px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;vertical-align:top;">ratio</td><td style="padding:5px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">reference cost divided by your cost; larger is better. The score is the geometric mean of three ratios.</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.55;max-width:100%;overflow-x:auto;margin:6px 0 14px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:2px;">About the figure: the general system map</div><table style="border-collapse:collapse;color:#111111;background:#ffffff;"><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">How to read it</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Read from left to right. The grey box is what you submit. Each coloured row is one task: its files, the judge that checks it, the cost it is given, and the ratio. The bar at the bottom combines the three ratios.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">What it shows</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">invsqrt is one file judged on every possible input; DQI is one very large file judged on 102,400 random shots; chemistry is a set of 16 files (15 small loader circuits and one table file) judged on error limits and an energy gap.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">So what</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">The three judges are different, so the three tasks fail in different ways: invsqrt can be proved by trying every input, DQI can only be sampled (so luck enters), chemistry has an accuracy budget that every change spends.</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.65;color:#111111;background:#ffffff;padding:2px 4px;"><b>Zoom 1: invsqrt.</b> The task is to give a quantum state a <i>phase</i> that depends on 1/&radic;x, where x is a 22-bit number held in the circuit. The circuit therefore has to compute an approximation of 1/&radic;x into spare wires, turn a few wires by the right angles, and then <b>give the spare wires back clean</b>, so that only the phase remains. The team ThemBones describes its design (they call it family C) in these steps:</div>

In [ ]:

def bx(ax, x, y, w, h, t, fc, ec=None, fs=8.4, tc="#111111", lw=1.4, bold=False):
    ax.add_patch(plt.Rectangle((x, y), w, h, fc=fc, ec=ec or fc, lw=lw, zorder=2))
    ax.text(x + w / 2, y + h / 2, t, ha="center", va="center", fontsize=fs, color=tc, zorder=3, fontweight="bold" if bold else "normal", linespacing=1.25)
def ar(ax, x1, y1, x2, y2, c="#555555"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", color=c, lw=1.5), zorder=1)

fig, ax = plt.subplots(figsize=(7, 4.3)); ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis("off"); ax.grid(False)
steps = [(2, 70, "1  input\nx: 22 bits", "#e9ecf2", "#14213d"), (36, 70, "2  shift\nnormalise x into\na fixed range", C[0], C[0]), (70, 70, "3  short square root\nin place", C[0], C[0]),
         (70, 38, "4  two correction\ndivisions", C[0], C[0]), (36, 38, "5  higher-order stage\non small table words", C[0], C[0]), (2, 38, "6  main divider\n(the digits of 1/sqrt x)", C[0], C[0]),
         (2, 6, "7  cyclic endpoint:\nrouter feeds 33\nZ rotations", C[1], C[1]), (36, 6, "8  undo everything\nin reverse; erase by\nmeasuring + repair", C[2], C[2])]
for x, y, t, fc, ec in steps:
    light = {C[0]: "#dbe8fb", C[1]: "#fbe3d8", C[2]: "#d8f1e6"}.get(fc, fc)
    bx(ax, x, y, 28, 24, t, light, ec, fs=8.2)
pairs = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7)]
ar(ax, 30, 82, 36, 82); ar(ax, 64, 82, 70, 82); ar(ax, 84, 70, 84, 62); ar(ax, 70, 50, 64, 50); ar(ax, 36, 50, 30, 50); ar(ax, 16, 38, 16, 30); ar(ax, 30, 18, 36, 18)
ax.text(70, 22, "result: the phase\nexp(2 pi i / sqrt x)\nis all that is left", fontsize=8.6, color="#111111", va="center", style="italic")
bx(ax, 2, 0, 96, 4.5, "all steps share 39 wires (37 in the newest file)", "#e9ecf2", "#14213d", fs=7.6)
headline(fig, "invsqrt: compute 1/sqrt(x), turn by 33 angles, give everything back", "Steps as described by team ThemBones (family C). Blue computes, orange turns, green cleans up.")
plt.show()


<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.55;max-width:100%;overflow-x:auto;margin:6px 0 14px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:2px;">About the figure: zoom on invsqrt</div><table style="border-collapse:collapse;color:#111111;background:#ffffff;"><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">How to read it</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Follow the arrows in the order of the numbers. Blue boxes compute, the orange box turns wires by angles, the green box gives everything back. The grey strip says how many wires the whole thing may use.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">What it shows</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Normalise the input by a shift, take a short square root in place, refine with two correction divisions and a small-table stage, divide, then a router feeds 33 rotations; everything computed is undone in reverse. All of it is fitted on 39 wires (37 in the newest file).</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">So what</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">The cost is wires times work, so the design question is how to reuse the same 39 wires for every block. The undo step is half of the work, which is why paying for it by measurement instead of Toffolis matters. Per-block costs are not published, so none are shown.</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.65;color:#111111;background:#ffffff;padding:2px 4px;"><b>Zoom 2: DQI.</b> The circuit is a decoder (the name comes from a quantum optimisation method). From the authors' notebooks, its heart is many small repeated pieces of finite-field arithmetic (for example inverses in a small field with 2<sup>6</sup> elements, about 110 of them on the 890-qubit line), a place to keep products so they are not recomputed, and a closing step in a large field. Its cost is the largest of the three, because it has by far the most gates.</div>

In [ ]:

def bx(ax, x, y, w, h, t, fc, ec=None, fs=8.4, tc="#111111", lw=1.4, bold=False):
    ax.add_patch(plt.Rectangle((x, y), w, h, fc=fc, ec=ec or fc, lw=lw, zorder=2))
    ax.text(x + w / 2, y + h / 2, t, ha="center", va="center", fontsize=fs, color=tc, zorder=3, fontweight="bold" if bold else "normal", linespacing=1.25)
def ar(ax, x1, y1, x2, y2, c="#555555"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", color=c, lw=1.5), zorder=1)

fig, ax = plt.subplots(figsize=(7, 4.4)); ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis("off"); ax.grid(False)
L = [("1  decoder input\n(a field element per round)", "#e9ecf2", "#14213d", 84, 12), ("2  many small repeated pieces:\nfinite-field inverses and products\n(110 on the 890-qubit line)", "#fbe3d8", C[1], 60, 20),
     ("3  products kept on idle wires\ninstead of recomputing them", "#fbe3d8", C[1], 42, 12), ("4  closing step in a large field", "#fbe3d8", C[1], 24, 12), ("5  measured clean-ups,\nthen the answer", "#d8f1e6", C[2], 4, 14)]
for i, (t, fc, ec, y0, h) in enumerate(L):
    bx(ax, 2, y0, 50, h, t, fc, ec, fs=8)
    if i < len(L) - 1: ar(ax, 27, y0, 27, L[i + 1][3] + L[i + 1][4])
bx(ax, 60, 76, 38, 18, "judge: 102,400 shots\nseed = SHA-256(file) + salt\n(same file, same shots)", "#f7f8fb", C[1], fs=8)
bx(ax, 60, 52, 38, 18, "checks: value errors = 0\nphase errors = 0", "#f7f8fb", C[1], fs=8)
bx(ax, 60, 28, 38, 20, "each failed shot raises the\nfactor R in the cost\n(one failed shot = +0.27%)", "#fff4d6", "#c98a00", fs=8)
for yy in (85, 61, 38): ar(ax, 52, 50, 60, yy, "#999999")
ax.text(60, 8, "343,998 Toffolis on 901 wires,\nall live about 91.7% of the run", fontsize=8.2, color="#111111", style="italic")
headline(fig, "DQI: the biggest circuit is judged by sampling, so luck and proof both matter", "Block names are from the authors' notebooks; the judge facts are from the official verifier.")
plt.show()


<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.55;max-width:100%;overflow-x:auto;margin:6px 0 14px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:2px;">About the figure: zoom on DQI</div><table style="border-collapse:collapse;color:#111111;background:#ffffff;"><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">How to read it</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">The left half is the circuit; the right half is its judge. The arrows on the left go top to bottom. The three small tags on the right are the three things the verifier checks and the one thing it multiplies the cost by.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">What it shows</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Repeated field-arithmetic pieces, a cache of products kept on idle wires, a closing step and measured clean-ups make up 343,998 Toffolis on 901 wires. The verifier draws 102,400 shots from a seed made of the file's hash; each failed shot raises the factor R.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">So what</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Because the judge only samples, a small edit that looks harmless can fail on shots the author never saw (our Part 8 results), and a file's luck is fixed by its hash. This is why proved edits are worth more than tested ones.</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.65;color:#111111;background:#ffffff;padding:2px 4px;"><b>Zoom 3: chemistry (dfthcsos).</b> The circuit estimates an energy of a large molecule. Most of its work is to <i>load numbers</i>: rotation angles are read out of a table by 15 small loader circuits, one for each chunk of the table. The table is stored in <code>submission.npz</code> together with the weights that define the problem. So the task has two kinds of file: <b>circuits</b> (the loaders) and <b>numbers</b> (the table and weights), and an improvement can come from either.</div>

In [ ]:

def bx(ax, x, y, w, h, t, fc, ec=None, fs=8.4, tc="#111111", lw=1.4, bold=False):
    ax.add_patch(plt.Rectangle((x, y), w, h, fc=fc, ec=ec or fc, lw=lw, zorder=2))
    ax.text(x + w / 2, y + h / 2, t, ha="center", va="center", fontsize=fs, color=tc, zorder=3, fontweight="bold" if bold else "normal", linespacing=1.25)
def ar(ax, x1, y1, x2, y2, c="#555555"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", color=c, lw=1.5), zorder=1)

fig, ax = plt.subplots(figsize=(7, 4.0)); ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis("off"); ax.grid(False)
bx(ax, 1, 40, 17, 28, "numbers:\nsubmission.npz\nweights W, H2_I,\nH1_I; angle words", "#e9ecf2", "#14213d", fs=7.6)
bx(ax, 24, 40, 19, 28, "15 loader circuits\nangle_k_0 ... 14\nread angle words\nfrom the table", "#d8f1e6", C[2], fs=7.8)
bx(ax, 49, 40, 19, 28, "rotations and\nenergy estimate\n13,434 Toffolis\n405 wires", "#dbe8fb", C[0], fs=7.8)
bx(ax, 74, 40, 25, 28, "judge:\nerror_1, error_2 <= 20 mHa\nenergy gap >= floor\nlambda about 46.9", "#f7f8fb", C[2], fs=7.6)
for a, b in ((18, 24), (43, 49), (68, 74)): ar(ax, a, 54, b, 54)
bx(ax, 1, 8, 40, 20, "refit / gauge polish:\nchanges the NUMBERS\n(about +0.002 points)\nspends accuracy budget", "#fbe3d8", C[1], fs=7.4)
bx(ax, 46, 8, 40, 20, "recompile a loader:\nchanges the CIRCUIT\n(about -6 Toffolis)\nsaves cost, same answer", "#fbe3d8", C[1], fs=7.4)
ar(ax, 10, 28, 10, 40, C[1]); ar(ax, 66, 28, 58, 40, C[1])
ax.text(50, 86, "cost = lambda x T x qubits   (one Toffoli of T = +0.0008 points)", fontsize=8.4, fontweight="bold", color="#111111", ha="center")
headline(fig, "chemistry: numbers feed loader circuits; two different kinds of change act on it", "Block names from the file layout of the public releases.")
plt.show()


<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.55;max-width:100%;overflow-x:auto;margin:6px 0 14px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:2px;">About the figure: zoom on chemistry</div><table style="border-collapse:collapse;color:#111111;background:#ffffff;"><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">How to read it</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Left to right is the data flow: tables, then loaders, then rotations, then the energy estimate, then the judge. The two orange tags under the arrows say which kind of change acts where.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">What it shows</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">The table and weights feed 15 loaders that produce the rotation angles; 13,434 Toffolis and 405 wires are paid for each run; the judge checks two reconstruction errors against a 20 mHa limit and an energy gap against a floor, and reads off lambda (about 46.9).</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">So what</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">A refit changes the numbers and spends accuracy budget (the limits are already active, so it is a polish worth about +0.002); recompiling a loader changes the circuit and saves Toffolis (about -6 in the 31.69 release). Deleting one loader Toffoli, in contrast, costs 4 to 6 Ha of error.</td></tr></table></div>

<a id="p3"></a>
<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;background:linear-gradient(120deg,#8e6bbf,#4a3280);color:#ffffff;padding:18px 24px;border-radius:12px;margin:22px 0 8px 0;"><div style="font-size:12.5px;letter-spacing:2px;text-transform:uppercase;color:#ffffff;">Part 3</div><div style="font-size:26px;font-weight:800;line-height:1.2;margin-top:4px;color:#ffffff;">How we got here: the public staircase</div><div style="font-size:15.5px;margin-top:6px;color:#ffffff;">One huge jump, many small steps, and a lot of luck-sized decimals</div></div>

In [ ]:
STEPS = [  # public board score, label (author of the notebook or release); none of the circuits is ours
 (24.65, "24.65\nsomejoint"), (24.94, "24.94\nalex chilton"), (28.06, "28.06\nAreeb Ali"), (28.14, "28.14\nAtakan Aldemir"), (28.15, "28.15\nAtakan Aldemir"),
 (28.24, "28.24\nKoushik / ozertuu"), (31.68, "31.68\nThemBones"), (31.69, "31.69\nAlmond Labs"), (32.85, "32.85\ntonylica"), (32.88, "32.88\nAreeb Ali"), (33.07, "33.07\nbest file per task"), (33.10, "33.10\ngoodpjw2008,\nboristown"), (33.14, "33.14\nmaoxin123")]
fig, ax = plt.subplots(figsize=(7, 4.4))
x = np.arange(len(STEPS)); y = [s[0] for s in STEPS]
ax.step(x, y, where="mid", color=C[0], lw=2.4); ax.scatter(x, y, color=C[0], zorder=3, s=36)
for xi, (yi, lab) in zip(x, STEPS): ax.text(xi, yi + (0.55 if xi % 2 == 0 else -1.25), lab, ha="center", fontsize=7.6, color=INK2)
ax.annotate("+3.4 in one day:\nnew circuits for\nall three tasks", xy=(5.5, 29.9), xytext=(1.2, 31.2), fontsize=10, color=C[1], arrowprops=dict(arrowstyle="->", color=C[1]))
ax.set_xticks([]); ax.set_ylabel("public board score"); ax.set_ylim(22.5, 36)
headline(fig, "ThemBones' release alone added +3.4 points", "Before it, 24.65 to 28.24 took two weeks of small gains; after it, the steps are tenths of a point.")
plt.show()


<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.55;max-width:100%;overflow-x:auto;margin:6px 0 14px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:2px;">About the figure: the staircase</div><table style="border-collapse:collapse;color:#111111;background:#ffffff;"><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">How to read it</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Left to right are public board scores in rising order; each dot is a notebook or release, labelled with its score and author. A steep step is a large jump.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">What it shows</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Most steps are a tenth of a point or less. One step, the ThemBones release, is +3.4 points: new circuits for all three tasks at once.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">So what</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Progress here comes in rare jumps from new designs, then in small polishes. A polish is worth a few thousandths; a new design is worth points. The ones at the top are tied to others' work, not to ours.</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.65;color:#111111;background:#ffffff;padding:2px 4px;"><b>Reading the staircase.</b> Between 24.65 and 28.24 many authors each added a little: a better invsqrt line, a merged decoder, a chemistry refit. On 20 November team <b>ThemBones</b> published new circuits for all three tasks and the board jumped by 3.4 points. Within hours almondlabs, prvsiyan and Almond Labs polished the chemistry and the DQI file (31.68 to 31.69). In the night to 21 November <b>tonylica</b> (a cheaper DQI and invsqrt), <b>Areeb Ali</b> (a better chemistry fit) and <b>Rithvik Redrouthu</b> (a 37-qubit invsqrt and a cheaper chemistry set) moved it to about 33. On 21 November <b>goodpjw2008</b> chained the chemistry loaders through <i>dead words</i> (table entries that no test can ever see) and combined the table of ozertuu with a loader of boristown, reaching <b>33.10186</b>; boristown's notebook of the same day, with a chemistry of his own (DAG splice, recompiled loader 6, FP64 refit), reports <b>33.10189</b> (our submission of his set scored 33.10185 on the board). Later the same day <b>maoxin123</b> recompiled all 15 chemistry loaders on the same words (3,375 instead of 3,396 Toffolis per pass) and reached <b>33.13514</b>. On 22 November <b>Areeb Ali</b> shortened four loaders (3,372 Toffolis per pass; 33.13869 on the board for our submission of his set), <b>varianceofx</b> and <b>mizeroluckygall</b> removed three T gates from the invsqrt file, and <b>goodpjw2008</b> combined these with a refitted table of <b>leotang09</b> (33.14394, the public best when we submitted it). On 22 and 23 November <b>maoxin123</b> brought the loaders to 3,357 Toffolis per pass (33.17), <b>goodpjw2008</b> added zero-test cuts to the DQI file (33.17067) and on 24 November <b>Areeb Ali</b> and <b>ozertuu</b> shortened the loaders again, the "26-Toffoli cut" (33.19326 on the board for our submission of their set). Later that day goodpjw2008, Areeb Ali, Mizero Lucky Gall and ozertuu published sets at about <b>33.2101</b> (shorter loaders, ten of them re-solved by goodpjw2008 and five by maoxin123, with a refitted table). Because each task is scored separately, a notebook can take the best file of each task from different authors: our own notebook submitted exactly that set and scored <b>33.21011</b> on the board. Nothing in this paragraph is ours except the choice of files.</div>

In [ ]:
slot("staircase", "A staircase built by many hands: most steps are small, one is huge")


<a id="p4"></a>
<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;background:linear-gradient(120deg,#0f9a68,#0a6a49);color:#ffffff;padding:18px 24px;border-radius:12px;margin:22px 0 8px 0;"><div style="font-size:12.5px;letter-spacing:2px;text-transform:uppercase;color:#ffffff;">Part 4</div><div style="font-size:26px;font-weight:800;line-height:1.2;margin-top:4px;color:#ffffff;">Ablation: what is responsible for what</div><div style="font-size:15.5px;margin-top:6px;color:#ffffff;">By task, by public step and by idea. This is arithmetic on published costs; we did not remove the ideas from the circuits and re-run them</div></div>

In [ ]:
names = list(SETS)
fig, ax = plt.subplots(figsize=(7, 4.2))
cols = {"invsqrt": C[0], "DQI": C[1], "chemistry": C[2]}
for i, (a, b) in enumerate(zip(names[:-1], names[1:])):
    lg = {"invsqrt": math.log(SETS[a]["invsqrt"] / SETS[b]["invsqrt"]), "DQI": math.log(SETS[a]["dqi"] / SETS[b]["dqi"]), "chemistry": math.log(SETS[a]["chem"] / SETS[b]["chem"])}
    tot = sum(lg.values()); pts = gm(SETS[b]) - gm(SETS[a]); left = 0
    for t, v in lg.items():
        w = pts * v / tot; ax.barh(i, w, left=left, color=cols[t], height=0.55, label=t if i == 0 else None)
        if w > 0.5: ax.text(left + w / 2, i, f"{t}\n{w:+.2f}", ha="center", va="center", fontsize=9.5, color="#ffffff", fontweight="bold")
        else: ax.text(left + w / 2, i + 0.42, f"{t} {w:+.2f}", ha="center", va="center", fontsize=8.5, color=INK)
        left += w
    ax.text(left + 0.04, i, f"{pts:+.2f} points", va="center", fontsize=11, color=INK)
ax.set_yticks(range(2)); ax.set_yticklabels([f"{a.split(' (')[0]}\n-> {b.split(' (')[0]}" for a, b in zip(names[:-1], names[1:])], fontsize=9.5)
ax.set_xlabel("points of the geometric mean (task shares from the log-ratios)"); ax.set_xlim(0, 4.6); ax.invert_yaxis()
headline(fig, "Jump 1 came from all three tasks, jump 2 from DQI", "Share of each step by task, computed from the authors' published costs.")
plt.show()
for a, b in zip(names[:-1], names[1:]):
    print(f"{a} -> {b}: invsqrt cost {SETS[a]['invsqrt']/SETS[b]['invsqrt']-1:+.1%} ratio, DQI {SETS[a]['dqi']/SETS[b]['dqi']-1:+.1%}, chemistry {SETS[a]['chem']/SETS[b]['chem']-1:+.1%}")


<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.55;max-width:100%;overflow-x:auto;margin:6px 0 14px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:2px;">About the figure: the attribution bars</div><table style="border-collapse:collapse;color:#111111;background:#ffffff;"><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">How to read it</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Each bar is one step of the staircase. Its length is the points gained; the coloured parts are the share of each task (blue invsqrt, orange DQI, green chemistry). Small parts are labelled below the bar.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">What it shows</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">From the 28.15 level to ThemBones all three tasks moved (invsqrt +1.70, chemistry +1.42, DQI +0.40). From ThemBones to the best public files the gain is mostly DQI (+0.94) and invsqrt (+0.37), with chemistry +0.12.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">So what</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Different phases are driven by different tasks, so the best place to look changes over time. The split is computed from published costs, not from re-running the circuits.</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.6;color:#111111;background:#eaf6f0;border-left:6px solid #0f9a68;border-radius:8px;padding:12px 18px;margin:10px 0;"><b>How these shares are computed.</b> The score is a geometric mean, so a task that gets 10% cheaper adds the same relative gain whichever task it is: score &times; (1.10<sup>1/3</sup> &minus; 1) &asymp; +3.3%. Each step is split between the tasks in proportion to the logarithm of each ratio. One cost per set is not published and is <i>derived</i> from the set's board score (DQI of the newest set, chemistry of the ThemBones set); the other numbers are the authors' own. The board score also contains a small random part (section 4).</div>

In [ ]:
IDEAS = pd.DataFrame([   # the five documented 31.69 ideas of Almond Labs; savings as they report them, valued with this notebook's price table
    ("LEND29: lend a wire (invsqrt)", "-0.5 Toffoli per shot", 0.5 * 0.0065),
    ("livescan + LDH: hold a product in an idle wire (DQI)", "-32.8 Toffolis", 32.8 * 0.000031),
    ("recompiled angle loaders (chemistry)", "-6 Toffolis", 6 * 0.00078),
    ("weight refit + gauge polish (chemistry)", "lambda_eff 46.9321 -> 46.9218", 0.0023)],
    columns=["idea", "saving reported by the authors", "points"])
tot = IDEAS.points.sum()
fig, ax = plt.subplots(figsize=(7, 3.2))
yy = np.arange(len(IDEAS))[::-1]; ax.barh(yy, IDEAS.points, color=[C[0], C[1], C[2], C[2]], height=0.55)
for yi, p in zip(yy, IDEAS.points): ax.text(p + 0.0002, yi, f"+{p:.4f}", va="center", fontsize=10.5)
ax.set_yticks(yy); ax.set_yticklabels(["LEND29\n(invsqrt)", "livescan + LDH\n(DQI)", "recompiled loaders\n(chemistry)", "refit + gauge\n(chemistry)"], fontsize=9.5); ax.set_xlim(0, 0.0058)
ax.set_xlabel("board points"); headline(fig, f"Four documented ideas add up to about +{tot:.4f} points", "Reported 31.67924 -> 31.69054 is +0.0113: the arithmetic and the board agree.")
plt.show(); print(IDEAS.to_string(index=False)); print("sum", round(tot, 4), "board step", round(31.69054 - 31.67924, 4))


<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.55;max-width:100%;overflow-x:auto;margin:6px 0 14px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:2px;">About the figure: the idea ablation</div><table style="border-collapse:collapse;color:#111111;background:#ffffff;"><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">How to read it</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">One bar per documented idea of the 31.69 release. The bar is the board points the idea is worth, from the savings its authors report and our price table.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">What it shows</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">The loaders (+0.0047) and the wire lending (+0.0032) are the largest; the refit and the idle-wire product are smaller. Together +0.0112, against +0.0113 on the board.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">So what</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">The agreement is a check on the price table. It also shows how small polishing ideas are: four of them are worth about a hundredth of a point, about 300 times less than the one new design that added 3.4.</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.65;color:#111111;background:#ffffff;padding:2px 4px;"><b>What the ablation says.</b> (1) Almost all the score came from <i>whole new circuits</i> (the staircase jump), not from polishing. (2) Polishing ideas are worth thousandths of a point each: the four documented 31.69 ideas together are worth about a hundredth. (3) The last two steps (31.69 to 33.10, +1.4) came from <i>other authors' better files</i>, a cheaper DQI and a 37-qubit invsqrt first, then a chain of chemistry loaders through dead words, not from refits. (4) The honest limit: this is arithmetic on reported savings. A true ablation would rebuild each circuit without one idea and run the verifier; we have not done that.</div>

<a id="p5"></a>
<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;background:linear-gradient(120deg,#c0392b,#7a1f17);color:#ffffff;padding:18px 24px;border-radius:12px;margin:22px 0 8px 0;"><div style="font-size:12.5px;letter-spacing:2px;text-transform:uppercase;color:#ffffff;">Part 5</div><div style="font-size:26px;font-weight:800;line-height:1.2;margin-top:4px;color:#ffffff;">Floors: how low can each lever go?</div><div style="font-size:15.5px;margin-top:6px;color:#ffffff;">What is proved, what is only tested, what is open, and how big the noise is</div></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;max-width:100%;overflow-x:auto;margin:8px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:3px 2px;">Floors and what is known about each (our tests are on the ThemBones and earlier files)</div><table style="border-collapse:collapse;font-size:14.5px;color:#111111;background:#ffffff;"><tr><th style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#dde6f3;overflow-wrap:anywhere;">Lever</th><th style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#dde6f3;overflow-wrap:anywhere;">Now (best public)</th><th style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#dde6f3;overflow-wrap:anywhere;">Hard floor we can state</th><th style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#dde6f3;overflow-wrap:anywhere;">What we tested</th><th style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#dde6f3;overflow-wrap:anywhere;">Status</th></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">invsqrt qubits</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">37 (Rithvik Redrouthu); 39 (ThemBones, tonylica)</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">at least 22: the input number must stay in place</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Removing the 39th wire of the 39-qubit file costs 1 Toffoli at each of 449 bits (about 449 vs a break-even of 42.5). A 37-qubit file exists, so a redesign works</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">open; the one lever that moved most</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">invsqrt Toffolis</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">1,339.9 per shot (tonylica, from the file); 1,379.4 (ThemBones)</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">none known</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">None of 1,493 Toffolis of the ThemBones file was removable by five exact rewrite families; no rotation or T gate droppable (LP optimum)</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">gate deletion closed; recompilation still pays (-0.5, -40)</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">DQI qubits</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">890 (older line), 901 (ThemBones)</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">none known</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">All wires are live about 91.7% of the run; peak equals all wires for 30.5M of 56.9M steps</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">no slack to compact</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">DQI Toffolis</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">343,998 (ThemBones)</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">none known</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">The two 'dead tail' removals failed the official verifier (12/18 and 20/14 value errors)</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">only proved edits are safe</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">chemistry Toffolis (T)</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">13,430 to 13,434 per pass</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">none known</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Exhaustive check of all 1,219 addresses: no removable Toffoli in the 15 loaders; a deletion costs 4.2 to 5.6 Ha of error against a 0.020 limit</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">recompilation only</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">chemistry lambda</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">about 46.92</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">set by three active limits</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">The refit sits where error 1, error 2 and the energy gap are all at their limits</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">polish, about +0.002; a smaller table (C 46 to 45) stalls at error_2 21.6 mHa, lambda above 48.2 (goodpjw2008)</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">board noise</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">about +-0.03 points</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">not removable</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">A failed DQI shot adds about 0.27% to the cost (-0.028 points); the board draws shots from a hash of the file</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">luck, not skill</td></tr></table></div>

In [ ]:
# The qubit-versus-Toffoli trade of invsqrt, and the price of one failed DQI shot, on one picture.
Q, TEQ = 39, 1616.796
fig, ax = plt.subplots(figsize=(7, 3.8))
extra = np.linspace(0, 80, 200)
ax.plot(extra, 100 * ((Q - 1) * (TEQ + extra) / (Q * TEQ) - 1), color=C[0], label="38 qubits, paying extra Toffoli-equivalents")
ax.axhline(0, color="#888888"); be = TEQ / (Q - 1)
ax.axvline(be, color=C[1], ls="--"); ax.text(be + 1, 1.6, f"break-even\n{be:.1f} Toffoli-eq", color=C[1], fontsize=10)
ax.set_xlabel("extra Toffoli-equivalents per shot paid to remove one qubit"); ax.set_ylabel("cost change, %")
ax.legend(loc="lower right"); headline(fig, "Removing a qubit pays only below about 42 Toffoli-equivalents", "Our minimal edit cost 449; a redesign (the 37-qubit file) got there.")
plt.show()


<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.55;max-width:100%;overflow-x:auto;margin:6px 0 14px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:2px;">About the figure: the qubit trade</div><table style="border-collapse:collapse;color:#111111;background:#ffffff;"><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">How to read it</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">The horizontal axis is how many extra Toffoli-equivalents per shot you pay to remove one invsqrt qubit; the vertical axis is the change of the cost in percent. Below zero is a gain.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">What it shows</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">The line crosses zero at about 42.5. Our minimal edit would have cost 449 Toffolis (far to the right, a loss of about 25%).</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">So what</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">A qubit is worth about 42 Toffolis, so it can only be removed by a redesign that costs less than that. The 37-qubit file shows such a redesign exists.</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.65;color:#111111;background:#ffffff;padding:2px 4px;"><b>Reading the table.</b> Only one floor is a hard fact (an in-place input needs its 22 qubits). Everything else is a <i>tested floor</i>: we searched in a defined way and found nothing, which says that single-gate deletions are exhausted, not that the circuit cannot get cheaper. The best evidence is the invsqrt: our test said "38 qubits needs a redesign", and another author then delivered 37 qubits. Floors move when someone changes the design, not when someone deletes a gate.</div>

<a id="p6"></a>
<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;background:linear-gradient(120deg,#e0a100,#a17300);color:#1f2430;padding:18px 24px;border-radius:12px;margin:22px 0 8px 0;"><div style="font-size:12.5px;letter-spacing:2px;text-transform:uppercase;color:#1f2430;">Part 6</div><div style="font-size:26px;font-weight:800;line-height:1.2;margin-top:4px;color:#1f2430;">Where is the best investment?</div><div style="font-size:15.5px;margin-top:6px;color:#1f2430;">Effort per point, measured in each task's own unit</div></div>

In [ ]:
LEVEL = 31.69
pts = 0.1; dc = 3 * pts / LEVEL                       # relative cost cut needed for +0.1 points (GM ~ cost^(-1/3))
need = pd.DataFrame([("invsqrt: Toffoli-equivalents", dc * 1616.8), ("invsqrt: qubits", dc * 39),
                     ("DQI: Toffolis", dc * 343998), ("DQI: qubits", dc * 901 / 2), ("chemistry: Toffolis of T", dc * 13434)], columns=["unit", "needed for +0.1 points"])
fig, ax = plt.subplots(figsize=(7, 3.6)); yy = np.arange(len(need))[::-1]
ax.barh(yy, need.iloc[:, 1], color=[C[0], C[0], C[1], C[1], C[2]], height=0.55); ax.set_xscale("log"); ax.set_xlim(0.1, 20000)
for yi, v in zip(yy, need.iloc[:, 1]): ax.text(v * 1.2, yi, f"{v:,.2f}" if v < 10 else f"{v:,.0f}", va="center", fontsize=10.5)
ax.set_yticks(yy); ax.set_yticklabels(need.iloc[:, 0], fontsize=10); ax.set_xlabel("how many units must be saved for +0.1 board points (log scale)")
headline(fig, "+0.1 points: 0.4 invsqrt qubits, or 127 chemistry or 3,300 DQI Toffolis", "Same score gain, very different effort.")
plt.show(); print(need.round(2).to_string(index=False))


<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.55;max-width:100%;overflow-x:auto;margin:6px 0 14px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:2px;">About the figure: the effort bars</div><table style="border-collapse:collapse;color:#111111;background:#ffffff;"><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">How to read it</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Each bar is how many units of that lever must be saved for +0.1 board points. The axis is logarithmic: each gridline is ten times more.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">What it shows</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">0.4 of an invsqrt qubit, 15 invsqrt Toffolis, 127 chemistry Toffolis, 4 DQI qubits or about 3,300 DQI Toffolis all give the same +0.1.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">So what</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">The same gain needs very different effort. DQI Toffolis are the most expensive way to gain a point; invsqrt qubits and Toffolis are the cheapest per unit,.</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;max-width:100%;overflow-x:auto;margin:8px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:3px 2px;">The investment table (prices at the 31.69 level; they grow in step with the score)</div><table style="border-collapse:collapse;font-size:14.5px;color:#111111;background:#ffffff;"><tr><th style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#dde6f3;overflow-wrap:anywhere;">Lever</th><th style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#dde6f3;overflow-wrap:anywhere;">Points per unit</th><th style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#dde6f3;overflow-wrap:anywhere;">Our verdict</th><th style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#dde6f3;overflow-wrap:anywhere;">Why</th></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">invsqrt qubit</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">+0.27</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">highest value per unit, already moved (37)</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">one qubit is 2.6% of the cost; a redesign was needed, not a deletion</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">invsqrt Toffoli-equivalent</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">+0.0065</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">pays only by recompilation</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">gate deletion closed by exhaustive tests; recompilation gave -0.5 and -40 per shot</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">DQI qubit</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">+0.023</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">unlikely</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">all wires live ~91.7% of the run</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">DQI Toffoli</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">+0.000031</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">needs thousands</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">single removals are unsafe: failed shots cost more than they save</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">chemistry Toffoli</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">+0.0008</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">recompilation</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">no loader gate removable; recompiled loaders gave -6</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">chemistry lambda refit</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">+0.001 per 0.005 of lambda</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">polished out</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">at the three active limits</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">file 'tickets' (zero-cost label changes)</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">+-0.03 noise</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">not a lever</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">re-rolls the board's random shots; the file is not better</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.6;color:#111111;background:#fff8e1;border-left:6px solid #e0a100;border-radius:8px;padding:12px 18px;margin:10px 0;"><b>Our recommendation.</b> Do not delete single gates any more: the exhaustive tests of Part 8 closed that road. Put effort into <b>re-designs that remove structure</b> (a wire, a repeated block, a table stored twice) and verify each with the official verifier, because that is where the 3.4-point jump and the 37-qubit invsqrt came from. Price every idea with the table above before building it, and never trust a 512-shot sample when the verifier draws 102,400.</div>

<a id="p7"></a>
<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;background:linear-gradient(120deg,#4285f4,#2f5faf);color:#ffffff;padding:18px 24px;border-radius:12px;margin:22px 0 8px 0;"><div style="font-size:12.5px;letter-spacing:2px;text-transform:uppercase;color:#ffffff;">Part 7</div><div style="font-size:26px;font-weight:800;line-height:1.2;margin-top:4px;color:#ffffff;">Human insight, AI search, and the hand-over</div><div style="font-size:15.5px;margin-top:6px;color:#ffffff;">Four worked examples: what the part is, how a person thinks, how that is handed to a machine, and where the machine still needs the person</div></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.65;color:#111111;background:#ffffff;padding:2px 4px;">There are two ways a better circuit is found. A <b>person</b> sees a reason: "this value is thrown away anyway", "this number repeats". A <b>search</b> has no reason, only a judge: it proposes an edit, the judge says whether the circuit is still correct and cheaper, and the search keeps what survives. Public work in this competition used both, usually together. The four examples below follow the same five questions, so you can see the pattern: <i>what the part is</i>, <i>the human's line of thought</i>, <i>the hand-over to the machine</i>, <i>what the machine adds</i>, and <i>where a human was still needed</i>.</div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.65;color:#111111;background:#ffffff;padding:2px 4px;"><h3 style="color:#111111;background:#ffffff;margin:14px 0 4px 0;">Example A &middot; Throwing away a temporary result for free</h3></div>

In [ ]:
# Measurement-based uncompute, exactly. Qubits a, b hold any state; t = a AND b is computed by ONE Toffoli.
# Instead of a second Toffoli to erase t, measure t in the X basis; if the result is 1 a phase (-1)^(a AND b) is left, repaired by one CZ.
rng = np.random.default_rng(1); psi = rng.normal(size=4) + 1j * rng.normal(size=4); psi /= np.linalg.norm(psi)   # state of (a, b)
ab = np.array([0, 0, 0, 1])                                   # a AND b for the basis states 00, 01, 10, 11
rows = []
for m in (0, 1):
    after = psi * ((-1.0) ** (m * ab))                        # X-basis outcome m leaves the phase (-1)^(m * a AND b)
    fixed = after * ((-1.0) ** (m * ab))                      # the CZ repair, applied only when m = 1
    rows.append((m, abs(np.vdot(psi, after)) ** 2, abs(np.vdot(psi, fixed)) ** 2))
print(pd.DataFrame(rows, columns=["measurement outcome", "overlap with the original, no repair", "overlap after the CZ repair"]).round(6).to_string(index=False))
print("Toffolis used: compute 1 + erase 0 = 1 (the plain method needs 2)")


<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.6;max-width:100%;overflow-x:auto;margin:12px 0;border-left:7px solid #2a9d8f;border-radius:8px;background:#ffffff;"><div style="font-size:18px;font-weight:800;color:#111111;background:#ffffff;padding:8px 12px 4px 12px;">A. Measure, then repair the phase</div><table style="border-collapse:collapse;color:#111111;background:#ffffff;width:100%;"><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">What the part is</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">Most circuits compute a temporary value (here the AND of two wires), use it, and have to erase it. Erasing normally costs a second Toffoli, the expensive gate. The code cell above shows the cheaper way on a random state.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">The human's line of thought</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">&ldquo;I no longer need this wire. Looking at it (measuring) destroys it for free. But looking leaves a trace: a sign flip that depends on the other wires. If I know exactly which sign flip, I can undo it with a cheap gate.&rdquo; The insight is about <i>information</i>: a measurement result is allowed to be random as long as the repair depends on it.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">The hand-over</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">The human states it as a rewrite rule that is <i>exact</i>: &ldquo;replace compute-then-uncompute by compute, measure in the X basis, and apply CZ if the outcome is 1&rdquo;, and gives the machine the judge: an exact simulation that must reproduce the original state for every measurement outcome.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">What the machine adds</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">It finds <b>all the places</b> where the rule is exact, in circuits with thousands of Toffolis (ThemBones report doing this wherever exact, sometimes inside a later block that computes the needed product anyway). A person would find the first few and give up.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">Where a human was still needed</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">To state the rule and its side condition (the repair must be conditional on the outcome), and to reject rewrites that look the same but are not exact. Our own &ldquo;dead tail&rdquo; edits (Part 8) are the counter-example: plausible, passed a 512-shot sample, and failed the verifier.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">Basis</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">Gidney, <a href="https://arxiv.org/abs/1709.06648" style="color:#0b3d91;background:transparent;">&ldquo;Halving the cost of quantum addition&rdquo;</a>, Quantum 2, 74 (2018): the temporary logical-AND is computed with Toffoli-class cost and erased by measurement plus a Clifford fix-up at no T cost.</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.65;color:#111111;background:#ffffff;padding:2px 4px;"><h3 style="color:#111111;background:#ffffff;margin:14px 0 4px 0;">Example B &middot; Getting 1/&radic;x: normalise first, then refine</h3></div>

In [ ]:
# The textbook route to 1/sqrt(x): (1) shift x by an even number of bits so that m = x / 4^k lies in [1, 4);
# then 1/sqrt(x) = 2^-k / sqrt(m); (2) refine a rough guess y by Newton steps y <- y * (3 - m*y*y) / 2, which roughly doubles the correct bits each time.
ms = np.linspace(1, 4, 4001); true = 1 / np.sqrt(ms); y = np.full_like(ms, 0.7); errs = []
for it in range(6):
    errs.append(np.max(np.abs(y / true - 1))); y = y * (3 - ms * y * y) / 2
errs.append(np.max(np.abs(y / true - 1)))
bits = [(-math.log2(e) if e > 0 else 52.0) for e in errs]
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(range(len(errs)), errs, marker="o", color=C[0]); ax.set_yscale("log"); ax.set_xlabel("number of Newton steps"); ax.set_ylabel("worst relative error (log scale)")
for i, (e, b) in enumerate(zip(errs, bits)):
    if i in (0, 2, 3, 4, 5, 6): ax.annotate(f"{max(b, 0):.0f} correct bits", (i, e), textcoords="offset points", xytext=(8, 6), fontsize=9.5)
headline(fig, "Each refinement step roughly doubles the correct bits", "Textbook method on m in [1, 4); the competition circuits use their own cheaper variants.")
plt.show(); print([f"{e:.2e}" for e in errs])


<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.55;max-width:100%;overflow-x:auto;margin:6px 0 14px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:2px;">About the figure: the refinement curve</div><table style="border-collapse:collapse;color:#111111;background:#ffffff;"><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">How to read it</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">The horizontal axis counts how many Newton steps have been applied; the vertical axis (logarithmic) is the worst error over all m between 1 and 4. A straight line on a log axis would be steady progress; a curve that bends down faster and faster means the error is squared at each step.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">What it shows</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Starting from one rough constant guess (0.7), the first steps are slow (about 1, 3 and 6 correct bits), then the error is squared each time: about 12 correct bits after four steps, 22 after five and 44 after six.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">So what</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">This is why the ThemBones design has only a few refinement stages (&ldquo;two correction divisions&rdquo;, &ldquo;a higher-order stage on small table words&rdquo;): few steps suffice if each one squares the error. The number of stages is the number of expensive multiplications.</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.6;max-width:100%;overflow-x:auto;margin:12px 0;border-left:7px solid #4285f4;border-radius:8px;background:#ffffff;"><div style="font-size:18px;font-weight:800;color:#111111;background:#ffffff;padding:8px 12px 4px 12px;">B. Normalise, take a short square root, refine</div><table style="border-collapse:collapse;color:#111111;background:#ffffff;width:100%;"><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">What the part is</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">The first half of the invsqrt circuit (steps 2 to 6 of the map in Part 2): bring x into a fixed range, get a rough 1/&radic;x, and sharpen it.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">The human's line of thought</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">&ldquo;Square roots of powers of 4 are powers of 2, so a shift removes the size of x for free. What is left lies in a short range, where a rough guess plus a few refinements is enough. Every refinement squares the error, so count the stages you really need.&rdquo; The reasoning is about <i>scale</i> and <i>convergence</i>.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">The hand-over</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">The human fixes the structure (shift, rough root, refinements) and a precise target: the phase error must stay below the tolerance for every one of the 3,133,452 inputs. The machine is given free parameters (table words, angles) and an exact judge.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">What the machine adds</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">It chooses the 33 rotation angles by a <b>linear program</b> that makes the worst-case margin as large as possible, trims Toffolis by exact rewrite passes, and packs the blocks onto 39 wires. These are numerical and combinatorial choices no one makes well by hand.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">Where a human was still needed</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">To choose the objective (largest margin, because the cost formula contains log2 of the margin), to decide that fewer wires matter more than fewer gates (Part 6), and to reject structures that cannot be fitted. Our own check that no rotation or T gate can be dropped says the angles are at the LP optimum: the human's structure, not the machine's polish, is where the next gain must come from.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">Basis</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">Newton&rsquo;s method and its quadratic convergence are standard numerical analysis; the circuit steps are as described by team ThemBones in their public notebook (credited at the end).</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.65;color:#111111;background:#ffffff;padding:2px 4px;"><h3 style="color:#111111;background:#ffffff;margin:14px 0 4px 0;">Example C &middot; Parking a result on a wire that is idle anyway</h3></div>

In [ ]:
# Schematic (NOT a real circuit): six wires over time. Dark bars = the wire is busy. Light gaps = idle. A product computed once can be parked on an idle wire
# and reused later instead of being recomputed. The scan that finds the gaps is what a program does well.
busy = {0: [(0, 100)], 1: [(0, 30), (62, 100)], 2: [(0, 18), (45, 58), (80, 100)], 3: [(10, 40), (72, 100)], 4: [(0, 12), (88, 100)], 5: [(0, 100)]}
fig, ax = plt.subplots(figsize=(7, 3.8))
for w, spans in busy.items():
    for a, b in spans: ax.barh(w, b - a, left=a, height=0.55, color="#8fa6c9")
ax.barh(4, 58, left=14, height=0.55, color=C[1]); ax.text(43, 4, "product parked", ha="center", va="center", fontsize=9.5, color="#ffffff", fontweight="bold")
for t_use in (46, 72): ax.annotate("reused here,\nnot recomputed", (t_use, 4.32), xytext=(t_use - 6, 6.0), fontsize=8.8, arrowprops=dict(arrowstyle="->", color=C[1]))
ax.annotate("computed once", (14, 4.3), xytext=(1, 5.9), fontsize=8.8, arrowprops=dict(arrowstyle="->", color=C[1]))
ax.set_yticks(range(6)); ax.set_yticklabels([f"wire {i + 1}" for i in range(6)]); ax.set_xlabel("time (steps of the run)"); ax.set_ylim(-0.6, 6.7); ax.invert_yaxis(); ax.grid(False)
headline(fig, "An idle wire can hold a result so it is never recomputed", "Schematic of the idea; the real scan covered 56.9 million steps and found 58,431 provably idle intervals (Almond Labs).")
plt.show()


<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.55;max-width:100%;overflow-x:auto;margin:6px 0 14px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:2px;">About the figure: the idle-wire schematic</div><table style="border-collapse:collapse;color:#111111;background:#ffffff;"><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">How to read it</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Each row is one wire, time runs left to right. Blue bars are times when a wire is in use; the white gaps are idle. The orange bar is a product kept on wire 5 during one of those gaps, and the arrows are the two later uses.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">What it shows</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Without parking, the product would be computed again at each use, at the price of Toffolis. With parking, it is computed once; the price is only that a wire must be free (and provably clean) for the whole interval.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">So what</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">No extra wire is needed, so the qubit cost stays the same while the Toffoli count falls. Finding which wires are provably free, and when, is a bookkeeping job over millions of steps: easy for a program, hopeless by eye.</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.6;max-width:100%;overflow-x:auto;margin:12px 0;border-left:7px solid #e0a100;border-radius:8px;background:#ffffff;"><div style="font-size:18px;font-weight:800;color:#111111;background:#ffffff;padding:8px 12px 4px 12px;">C. Remember a product instead of recomputing it</div><table style="border-collapse:collapse;color:#111111;background:#ffffff;width:100%;"><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">What the part is</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">In the DQI decoder the same product of field elements is needed again and again. The idea is a cache: compute it once, keep it, reuse it.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">The human's line of thought</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">&ldquo;Never do the same work twice, and use space that would otherwise be empty.&rdquo; This is the human habit of <i>memoisation</i> plus an eye for free space; it needs no quantum insight.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">The hand-over</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">The human defines what &ldquo;safe&rdquo; means: a wire may be borrowed only if it is <i>provably zero</i> for the whole interval and is returned zero. That sentence is the judge&rsquo;s rule.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">What the machine adds</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">A liveness scan over the whole run (Almond Labs&rsquo; <code>livescan</code>) lists every provably idle interval (58,431) and a placement step (<code>LDH</code>) parks a repeated product in one of them: about 32.8 Toffolis per shot saved, 0 value and 0 phase errors on both local verifier salts, as the authors report.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">Where a human was still needed</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">To define &ldquo;provably&rdquo;. A wire that looks idle on a sample can be alive on the verifier&rsquo;s shots: our &ldquo;dead tail&rdquo; removals were exactly this mistake and cost 12 to 20 value errors (Part 8). The human must keep the judge honest.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">Basis</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">Public notebooks of Almond Labs (livescan, LDH) and of grantprater (product-chain caching, credited at the end); the schematic above is ours.</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.65;color:#111111;background:#ffffff;padding:2px 4px;"><h3 style="color:#111111;background:#ffffff;margin:14px 0 4px 0;">Example D &middot; Rotating the table without changing the physics</h3></div>

In [ ]:
# Toy version of the chemistry 'gauge polish'. A table A is only used through M = A A^T, so any rotation A -> A Q (Q orthogonal) leaves M unchanged.
# The cost depends on the sum of the absolute values (a one-norm), which DOES change with Q. A small optimiser picks the best Q.
from scipy.linalg import expm
from scipy.optimize import minimize
rng = np.random.default_rng(7); A = rng.normal(size=(12, 4)) * np.array([3.0, 1.5, 0.7, 0.3]); M = A @ A.T
def rot(p):
    S = np.zeros((4, 4)); S[np.triu_indices(4, 1)] = p; return expm(S - S.T)
f = lambda p: np.abs(A @ rot(p)).sum()
best = min((minimize(f, rng.normal(size=6) * 0.8, method="Powell") for _ in range(8)), key=lambda r: r.fun)
B = A @ rot(best.x)
print(f"one-norm before {np.abs(A).sum():.3f}  after {np.abs(B).sum():.3f}  ({np.abs(B).sum() / np.abs(A).sum() - 1:+.1%})")
print(f"largest change of M = A A^T: {np.abs(B @ B.T - M).max():.2e}  (zero up to rounding: the physics is unchanged)")
fig, axs = plt.subplots(1, 2, figsize=(7, 3.2))
for ax, X, ttl in zip(axs, (A, B), ("before the rotation", "after the rotation")):
    ax.imshow(np.abs(X), cmap="Blues", aspect="auto", vmin=0, vmax=np.abs(A).max()); ax.set_title(ttl, fontsize=11); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
headline(fig, "Same M, smaller entries: the table is rotated, not changed", "Darker cell = larger entry. Toy numbers, not the real table.")
plt.show()


<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.55;max-width:100%;overflow-x:auto;margin:6px 0 14px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:2px;">About the figure: the rotation toy</div><table style="border-collapse:collapse;color:#111111;background:#ffffff;"><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">How to read it</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Each picture is the same table of numbers drawn as a heat map (darker means a larger entry). The left is the original table; the right is the same table after the best rotation found by a small optimiser.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">What it shows</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">The product M that the physics uses is identical to rounding error, but the total size of the entries has gone down, and the cost formula is driven by that total.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">So what</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">A rotation is a free choice: it spends no accuracy budget. This is why the weight refit and gauge polish gave a small, safe gain in the 31.69 release (+0.002).</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.6;max-width:100%;overflow-x:auto;margin:12px 0;border-left:7px solid #c96442;border-radius:8px;background:#ffffff;"><div style="font-size:18px;font-weight:800;color:#111111;background:#ffffff;padding:8px 12px 4px 12px;">D. Choose the best rotation of the table</div><table style="border-collapse:collapse;color:#111111;background:#ffffff;width:100%;"><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">What the part is</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">The chemistry task stores its problem as tables of numbers. The cost grows with the total size (a one-norm) of those numbers. The tables are used only through products, so many different tables describe the same physics.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">The human's line of thought</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">&ldquo;If the answer does not depend on how I orient the table, I can orient it for the cheapest cost.&rdquo; This is the symmetry argument of a physicist: a freedom of description (a <i>gauge</i>) can be used.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">The hand-over</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">The human writes the optimisation: minimise the cost factor lambda while keeping three accuracy limits (two reconstruction errors at most 20 mHa and an energy gap above its floor). The machine receives a smooth objective and exact constraint checks.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">What the machine adds</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">A constrained optimiser (L-BFGS style, refit of weights and an orthogonal gauge) walks a space with thousands of numbers and lowers lambda from 46.9321 to 46.9218 in the 31.69 release; later authors polished further.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">Where a human was still needed</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">To notice that the freedom exists, to fix the limits as hard constraints (a wrong sign gives a fast but wrong table), and to recognise the end of the road: the three limits are active, so the refit is a polish now and the next gain has to come from the circuits, not from the numbers.</td></tr><tr><td style="padding:7px 10px;border:1px solid #c9d2e0;background:#e8eef8;color:#111111;font-weight:700;vertical-align:top;width:150px;">Basis</td><td style="padding:7px 12px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;vertical-align:top;">Lee, Berry, Gidney et al., <a href="https://doi.org/10.1103/PRXQuantum.2.030305" style="color:#0b3d91;background:transparent;">PRX Quantum 2, 030305 (2021)</a> (tensor hypercontraction for chemistry on a quantum computer, including the FeMoCo molecule); the refit and gauge steps are as described in the public notebooks of almondlabs and Almond Labs.</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.65;color:#111111;background:#ffffff;padding:2px 4px;"><h3 style="color:#111111;background:#ffffff;margin:14px 0 4px 0;">How the hand-over works</h3></div>

In [ ]:
def bx(ax, x, y, w, h, t, fc, ec=None, fs=8.4, tc="#111111", lw=1.4, bold=False):
    ax.add_patch(plt.Rectangle((x, y), w, h, fc=fc, ec=ec or fc, lw=lw, zorder=2))
    ax.text(x + w / 2, y + h / 2, t, ha="center", va="center", fontsize=fs, color=tc, zorder=3, fontweight="bold" if bold else "normal", linespacing=1.25)
def ar(ax, x1, y1, x2, y2, c="#555555"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", color=c, lw=1.5), zorder=1)
fig, ax = plt.subplots(figsize=(7, 4.6)); ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis("off"); ax.grid(False)
ax.text(2, 96, "HUMAN", fontsize=11, fontweight="bold", color=C[0]); ax.text(66, 96, "AI SEARCH", fontsize=11, fontweight="bold", color=C[1])
H = ["sets the goal and prices it\n(a qubit = 42 Toffolis)", "builds the judge that cannot be fooled\n(all inputs, many salts)", "says what an edit may touch\n(accuracy limits, safe wires)", "reads the failures and the luck\n(dead tail, file tickets)"]
Aa = ["proposes thousands of edits", "scans the whole circuit\n(56.9M steps)", "solves the LP and the refit", "tries variants in parallel\nand never tires"]
for i, t in enumerate(H): bx(ax, 2, 74 - i * 19, 34, 15, t, "#dbe8fb", C[0], fs=7.6)
for i, t in enumerate(Aa): bx(ax, 64, 74 - i * 19, 34, 15, t, "#fbe3d8", C[1], fs=7.6)
bx(ax, 40, 40, 20, 22, "THE LOOP\nedit -> judge ->\nkeep or reject", "#e9ecf2", "#14213d", fs=7.8, bold=True)
for i in range(4): ar(ax, 36, 81.5 - i * 19, 40, 51); ar(ax, 60, 51, 64, 81.5 - i * 19, "#999999")
ax.text(50, 6, "the judge decides what survives;\nthe human decides what the judge asks", ha="center", fontsize=8.6, style="italic", color=INK)
headline(fig, "The machine searches; the human chooses what counts", "Division of labour as seen in the public work on this task.")
plt.show()


<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.55;max-width:100%;overflow-x:auto;margin:6px 0 14px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:2px;">About the figure: the hand-over diagram</div><table style="border-collapse:collapse;color:#111111;background:#ffffff;"><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">How to read it</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Left are the jobs we saw humans do, right are the jobs the search does. The grey box in the middle is the loop they share: an edit is proposed, the judge decides, only accepted edits are kept.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">What it shows</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">Every job on the left is about meaning (what is worth doing, what is correct, what is luck); every job on the right is about scale (how many edits, how long a circuit, how many parameters).</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#dde6f3;color:#111111;font-weight:700;white-space:nowrap;vertical-align:top;">So what</td><td style="padding:6px 10px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;">A search without a good judge optimises the judge, not the goal. This is what happened with the file tickets (a different hash re-rolls the random shots: the score moves, the circuit is not better). A human has to notice it.</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;max-width:100%;overflow-x:auto;margin:8px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:3px 2px;">Six places where a person was still needed</div><table style="border-collapse:collapse;font-size:14.5px;color:#111111;background:#ffffff;"><tr><th style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#dde6f3;overflow-wrap:anywhere;">Where the AI still needed a human</th><th style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#dde6f3;overflow-wrap:anywhere;">Evidence in this notebook</th></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Deciding what is worth searching for</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Part 6: one invsqrt qubit equals about 42 Toffolis, so a search for single-gate deletions was the wrong search; a redesign (the 37-qubit file) was the right one.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Building a judge that cannot be fooled</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Part 8: two DQI edits passed a 512-shot sample and failed the official 102,400-shot verifier with 12 to 20 value errors.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Telling an improvement from luck</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Part 4: the board draws DQI shots from the file&rsquo;s hash, so a zero-cost label re-rolls them; a single failed shot moves the score by about 0.03.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Fixing what must not change</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Part 8: removing one chemistry loader Toffoli gives 4.2 to 5.6 Ha error against a 0.020 limit; the accuracy limits must be hard constraints.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Knowing when a road is closed</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Part 8: exhaustive checks (all 3,133,452 inputs, all 1,219 addresses) showed no gate left to delete; the effort was redirected.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Explaining and crediting</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">A search cannot say who built which part; the credit tables of this page and of the repository were written by reading the authors&rsquo; notebooks.</td></tr></table></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;max-width:100%;overflow-x:auto;margin:8px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:3px 2px;">Research basis (every entry was checked against its journal or arXiv page; none is a claim of ours)</div><table style="border-collapse:collapse;font-size:14.5px;color:#111111;background:#ffffff;"><tr><th style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#dde6f3;overflow-wrap:anywhere;">Source</th><th style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#dde6f3;overflow-wrap:anywhere;">What it shows</th><th style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#dde6f3;overflow-wrap:anywhere;">How we use it</th></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;"><a href="https://arxiv.org/abs/1709.06648" style="color:#0b3d91;background:transparent;">Gidney, Quantum 2, 74 (2018)</a></td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">A temporary AND costs Toffoli-class gates to compute and none to erase (measurement plus Clifford fix-up).</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Example A; the cost rule that lets many circuits drop their uncompute Toffolis.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;"><a href="https://arxiv.org/abs/2408.08292" style="color:#0b3d91;background:transparent;">Jordan et al., Nature 646, 831 (2025)</a></td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Decoded Quantum Interferometry turns an optimisation problem into a decoding problem.</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Background of the DQI task (Part 2).</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;"><a href="https://doi.org/10.1103/PRXQuantum.2.030305" style="color:#0b3d91;background:transparent;">Lee et al., PRX Quantum 2, 030305 (2021)</a></td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Tensor hypercontraction makes chemistry circuits cheaper, with FeMoCo as the example.</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Background of the chemistry task; Example D.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;"><a href="https://doi.org/10.1038/s41586-023-06924-6" style="color:#0b3d91;background:transparent;">Romera-Paredes et al. (FunSearch), Nature 625, 468 (2024)</a></td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">A language model proposes programs and a systematic evaluator keeps the good ones.</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">The same pattern as our loop: the evaluator, not the proposer, defines what is true.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;"><a href="https://arxiv.org/abs/2402.14396" style="color:#0b3d91;background:transparent;">Ruiz et al. (AlphaTensor-Quantum), Nature Machine Intelligence 7, 374 (2025)</a></td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Reinforcement learning lowers T-count, matches the best human-designed solutions for arithmetic and chemistry, and takes about eight hours per circuit.</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Evidence that search can rediscover and sometimes beat human constructions, at a compute price.</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Khattar et al., arXiv:2610.23832 (as cited in the public notebooks of this competition)</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">The task and scoring documents of the competition.</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">We cite it as the public notebooks do; we did not re-read it for this page.</td></tr></table></div>

In [ ]:
slot("ladder", "Learning step by step, side by side with an AI helper")


<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;max-width:100%;overflow-x:auto;margin:8px 0;"><div style="font-size:15px;font-weight:700;color:#111111;background:#ffffff;padding:3px 2px;">A ladder of six exercises</div><table style="border-collapse:collapse;font-size:14.5px;color:#111111;background:#ffffff;"><tr><th style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#dde6f3;overflow-wrap:anywhere;">Step</th><th style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#dde6f3;overflow-wrap:anywhere;">Do this</th><th style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#dde6f3;overflow-wrap:anywhere;">You learn</th></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">1</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Read the cost formulas of Part 1 and recompute the ThemBones costs with a calculator.</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">what the score measures</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">2</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Run Example A and change the random seed.</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">why measuring can replace a gate</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">3</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Run Example B and change the starting guess (0.7); find which guesses fail.</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">when a refinement converges</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">4</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Take the price table of Part 6 and rank three imaginary ideas before touching a circuit.</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">price before build</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">5</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Download a public circuit, count its gates by type, and find which gate dominates.</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">where the cost lives</td></tr><tr><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">6</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">Write an exact test for one rewrite on the whole input domain, and run the official verifier on the result.</td><td style="padding:6px 10px;border:1px solid #c9d2e0;text-align:left;color:#111111;background:#ffffff;overflow-wrap:anywhere;vertical-align:top;">a judge that cannot be fooled</td></tr></table></div>

<a id="p8"></a>
<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;background:linear-gradient(120deg,#c96442,#8a3a22);color:#ffffff;padding:18px 24px;border-radius:12px;margin:22px 0 8px 0;"><div style="font-size:12.5px;letter-spacing:2px;text-transform:uppercase;color:#ffffff;">Part 8</div><div style="font-size:26px;font-weight:800;line-height:1.2;margin-top:4px;color:#ffffff;">What we tried, what failed and why</div><div style="font-size:15.5px;margin-top:6px;color:#ffffff;">Our experiments of 19 and 20 November 2026 on the public ThemBones files, with numbers</div></div>

In [ ]:
slot("doors", "Most doors stay shut and many rooms are empty; trying them is how the open one is found")


<a id="context"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15.5px; line-height: 1.55; color: #111111; background: #fff7e0; border-left: 6px solid #c98a00; padding: 11px 16px; border-radius: 8px; margin-top: 10px;">
<b>Context (20 November 2026).</b> The public board moved from <b>28.24</b> to <b>31.68</b> on 20 November, when team ThemBones (Danish Ali) published their circuits under Apache-2.0 and almondlabs improved the chemistry part (31.68159). This notebook's earlier version re-packaged that public file set and scored <b>31.68159</b> (submission 58626935), a tie with the public notebook. <b>None of the circuits is ours.</b> The experiments below were run on those public files by the notebook's author with several AI agents. The official verifiers ran on Kaggle (one DQI run takes about 90 to 110 seconds); invsqrt and the chemistry loaders were checked with our own exact simulators. This notebook writes no submission file: it only reports numbers.
</div>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15.5px; line-height: 1.55; color: #111111; background: #fff7e0; border-left: 6px solid #c98a00; padding: 11px 16px; border-radius: 8px; margin-top: 10px;"><b>About this version (20 November 2026).</b> New: this page is the written record of the experiments of 19 and 20 November (it replaces the earlier version that only ran one DQI test). Not covered here: the next experiments, and any gain; none was found. All numbers are measured and listed in the results table.</div>

<a id="results"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15.5px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #c9d6ea; border-radius: 10px; padding: 14px 18px; margin-top: 10px;">
<div style="font-size: 19px; font-weight: 700; color: #0b1f4b; margin-bottom: 6px; background: #ffffff;">Results at a glance</div>
The next cell prints one row per experiment. "Verdict" is about the score only: every experiment taught us something, none of them gave a better file.
</div>

In [ ]:
RESULTS = pd.DataFrame([
    ("1", "DQI", "Port the 19-Toffoli inverse leaf to the 110th GF(2^6) site", "official verifier", "2 value errors; cost 348.04e9 vs 345.11e9", "failed"),
    ("2", "DQI", "Delete 'dead tail' gates: cascade96 (-96 Toffolis), void2 (-2)", "official verifier, 2 salts", "12/18 and 20/14 value errors; cost up 3.6% to 5.3%", "failed"),
    ("3", "invsqrt", "Drop a rotation, a T gate, a Toffoli or a qubit of the 39-qubit file", "exact simulation, all 3,133,452 inputs", "0 of 33 rotations, 0 of 1,493 Toffolis; 38 qubits costs 449 Toffolis", "nothing found"),
    ("4", "DQI", "Free a wire of the 890-qubit line by re-assignment", "liveness census of the whole run", "peak 890 for 30.5M of 56.9M steps; wires live ~91.7%", "nothing found"),
    ("5", "chemistry", "Remove a Toffoli from the 15 loaders", "exhaustive, all 1,219 addresses, both directions", "0 of 3,467 Toffolis removable; one deletion = 4.2-5.6 Ha error", "nothing found"),
], columns=["#", "task", "what we tried", "how it was checked", "result", "verdict"]).set_index("#")
pd.set_option("display.max_colwidth", None); pd.set_option("display.width", 250)
display(RESULTS.style.set_properties(**{"text-align": "left", "color": "#111111", "background-color": "#ffffff", "font-size": "14px"})
        .set_table_styles([{"selector": "th", "props": [("text-align", "left"), ("color", "#0b1f4b"), ("background-color", "#e8eef8")]}]))

In [ ]:
# First-screen figure: what each DQI edit promised and what the official verifier measured.
d = RUNS[RUNS["Toffolis removed"] > 0].reset_index(drop=True)
MEAN_T = {"890 q": 365_738.81, "901 q": 343_998.1}         # mean Toffolis per shot of the two base files (official verifier numbers)
d["claimed %"] = [-100 * r / MEAN_T[b] for r, b in zip(d["Toffolis removed"], d["base"])]
lab = [f"{c}\n({s})" for c, s in zip(d["candidate"].str.replace(" (HW-31 leaf)", "", regex=False), d["salt"])]
fig, ax = plt.subplots(2, 1, figsize=(7, 8.6))
y = np.arange(len(d))[::-1]
ax[0].barh(y, d["value errors"], color=C[7], height=0.6)
for yi, v in zip(y, d["value errors"]): ax[0].text(v + 0.3, yi, f"{v}", va="center", fontsize=11.5, color=INK, fontweight="bold")
ax[0].set_yticks(y); ax[0].set_yticklabels(lab); ax[0].set_xlim(0, 24)
ax[0].set_xlabel(f"value errors in {SHOTS:,} shots (base files: 0)"); ax[0].set_title("Official verifier: failed shots", loc="left")
ax[1].barh(y + 0.2, d["claimed %"], color=C[2], height=0.36, label="claimed (fewer Toffolis)")
ax[1].barh(y - 0.2, d["cost change %"], color=C[7], height=0.36, label="measured (official cost)")
for yi, a, b in zip(y, d["claimed %"], d["cost change %"]):
    ax[1].text(0.08, yi + 0.2, f"{a:+.4f}%", va="center", fontsize=10.5, color=INK)
    ax[1].text(b + 0.08, yi - 0.2, f"{b:+.2f}%", va="center", fontsize=10.5, color=INK, fontweight="bold")
ax[1].axvline(0, color="#888888", lw=1); ax[1].set_yticks(y); ax[1].set_yticklabels(lab); ax[1].set_xlim(-0.6, 7)
ax[1].set_xlabel("change of the DQI cost, % (lower is better)"); ax[1].set_title("Claimed saving vs measured result", loc="left"); ax[1].legend(loc="upper right", bbox_to_anchor=(1.0, 0.99)); ax[1].set_ylim(-0.6, 5.3)
headline(fig, "Every DQI edit made the cost worse: a few failed shots outweigh the saved gates",
         "Savings of 1 to 96 Toffolis (at most 0.03% of cost) against 2 to 20 failed shots (0.8% to 5.3% of cost).")
plt.show()
print(d[["candidate", "salt", "Toffolis removed", "value errors", "cost", "base cost", "cost change %"]].round(3).to_string(index=False))

<a id="e1"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15.5px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #c9d6ea; border-radius: 10px; padding: 14px 18px; margin-top: 10px;">
<div style="font-size: 19px; font-weight: 700; color: #0b1f4b; margin-bottom: 6px; background: #ffffff;">1. DQI: the 110th GF(2<sup>6</sup>) inverse site</div>
<table style="border-collapse: collapse; background: #ffffff; color: #111111; font-size: 15.5px;"><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Idea</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">The 890-qubit public DQI line contains 110 inverse blocks over GF(2<sup>6</sup>), 20 Toffolis each. The HW-31 leaf by manwithacat rewrites 100 of them to 19 Toffolis (718 operations become 1,058, with history caching and free phase repayment); the leaf-port file of goodpjw2008 rewrites 109. One site was left. We applied the same leaf there, with wires and bits mapped from a template site of the same shape. All 109 existing templates were consistent with each other.</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Test</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">The unchanged official verifier on Kaggle (84 seconds), 102,400 shots.</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Result</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;"><b>2 value errors.</b> The repetition factor rose to 1.2014 and the cost to <b>348.04e9 against 345.11e9</b> for the file without the edit. The static count was 1 Toffoli lower, exactly as planned.</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Why it failed</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">The 110th site is not clean on entry: a wire that is zero at the other 109 sites carries a value there on some shots. The mapping reproduced the shape of the block, but shape says nothing about the state that arrives.</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">What it tells us</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">The authors before us most likely left this site alone for a reason. Before porting a rewrite to a new site, prove its entry conditions on that site; 109 agreeing templates are not evidence for the 110th.</td></tr></table>
</div>

<a id="e2"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15.5px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #c9d6ea; border-radius: 10px; padding: 14px 18px; margin-top: 10px;">
<div style="font-size: 19px; font-weight: 700; color: #0b1f4b; margin-bottom: 6px; background: #ffffff;">2. DQI: the "dead tail" candidates cascade96 and void2</div>
<table style="border-collapse: collapse; background: #ffffff; color: #111111; font-size: 15.5px;"><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Idea</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">On the ThemBones 901-qubit DQI file a public analysis pass reported two Toffolis whose product is identically zero. If that holds, the two gates are the identity and a tail of gates that depends on them can go too. Two candidates: <b>void2</b> removes only the two gates (-2 Toffolis), <b>cascade96</b> removes the whole tail (-96 Toffolis, about +0.003 points if correct).</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Test</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">Local simulation of 512 legal shots first (no difference seen). Then the official verifier on Kaggle, about 111 seconds per run, with two salts (two different board draws).</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Result</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;"><b>cascade96: 12 and 18 value errors</b>, repetition factor 1.2347 and 1.2500, cost 3.447e11 and 3.490e11. <b>void2: 20 and 14 value errors</b>, cost 3.504e11 and 3.463e11. The unchanged file: <b>0 errors, cost 3.327e11</b>.</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Why it failed</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">The two gates are not the identity. The "zero product" claim came from the tool's symbol algebra and was never proved on the distribution of inputs that the verifier draws (or its precondition does not hold on those shots). Our 512 sampled shots proved nothing, because the flags involved are almost never 1: a sample that never triggers the gate cannot show what the gate does.</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">What it tells us</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">void2 is the useful control: it removes only the two suspect gates and still fails, so the fault is in the claim and not in the cascade built on it. Test the smallest possible edit first, and treat "never fired in my sample" as a question, not as an answer.</td></tr></table>
</div>

<a id="e3"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15.5px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #c9d6ea; border-radius: 10px; padding: 14px 18px; margin-top: 10px;">
<div style="font-size: 19px; font-weight: 700; color: #0b1f4b; margin-bottom: 6px; background: #ffffff;">3. invsqrt: the 39-qubit file is tight in angles, gates and wires</div>
<table style="border-collapse: collapse; background: #ffffff; color: #111111; font-size: 15.5px;"><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Idea</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">The ThemBones 39-qubit invsqrt costs 63,055.04. We looked for any exact local saving: better rotation angles, one rotation or T gate fewer, one Toffoli fewer, or one qubit fewer.</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Test</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">Our own bit-packed simulator over <b>all 3,133,452 inputs</b> (it reproduces the published figures), a linear program for the angles, and five exact rewrite scans over all 1,493 Toffoli-like gates.</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Result</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;"><b>Angles:</b> at the LP optimum (slack 51,917,278 units against an optimum of 51,917,281; the 3 units are rounding). <b>Rotations:</b> none of the 33 can be dropped (every LP is infeasible; the closest, rotation 17, is short by 4.8e7 units), and merging rotation 17 into any of 32 partners is infeasible too. <b>T gates:</b> the two exact T gates on qubit 21 cannot be dropped. <b>Toffolis:</b> none of 1,493 is removable by constant or implied control, affine product, same-target merge, measurement uncompute, or zero-operand CSWAP. <b>Qubits:</b> all 39 wires are occupied at 3,012 positions in 449 separate runs.</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Why it failed</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">The 39th wire is the second carry of the adders. Removing it costs one extra Toffoli at each of 449 bits, while one qubit is worth only about 42 Toffoli-equivalents (figure below). The file is already what the linear program and the exact scans would produce.</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">What it tells us</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">A 38-qubit invsqrt needs a redesign (a long-lived divider value would have to be removed across about 15,000 instructions), not a local edit.</td></tr></table>
</div>

In [ ]:
# The qubit-vs-Toffoli trade of the 39-qubit invsqrt. cost = qubits x Toffoli-equivalents per shot.
Q, COST = 39, 63055.04
TEQ = COST / Q                                   # Toffoli-equivalents per shot (1616.8)
BREAK_EVEN = TEQ / (Q - 1)                       # extra Toffoli-equivalents that going from 39 to 38 qubits can pay for (about 42.5, exact)
NEED = 449                                       # one extra Toffoli at each of 449 bits
x = np.linspace(0, 500, 501)
rel = ((Q - 1) * (TEQ + x) / COST - 1) * 100     # cost change of a 38-qubit file that needs x extra Toffoli-equivalents
fig, ax = plt.subplots(figsize=(7, 4.6))
ax.plot(x, rel, color=C[0]); ax.axhline(0, color="#888888", lw=1)
ax.fill_between(x, rel, 0, where=rel < 0, color=C[2], alpha=0.25); ax.fill_between(x, rel, 0, where=rel > 0, color=C[7], alpha=0.12)
ax.scatter([NEED], [((Q - 1) * (TEQ + NEED) / COST - 1) * 100], color=C[7], zorder=5, s=70)
ax.annotate(f"what the 39th wire costs to remove:\n+{NEED} Toffolis -> cost {((Q - 1) * (TEQ + NEED) / COST - 1) * 100:+.1f}%", (NEED, ((Q - 1) * (TEQ + NEED) / COST - 1) * 100),
            xytext=(235, 20), fontsize=11, color=INK, arrowprops=dict(arrowstyle="->", color=INK2))
ax.annotate(f"break-even: about {BREAK_EVEN:.1f} Toffoli-eq", (BREAK_EVEN, 0), xytext=(70, -2.2), fontsize=11, color=INK, arrowprops=dict(arrowstyle="->", color=INK2))
ax.set_xlabel("extra Toffoli-equivalents per shot paid for removing one qubit"); ax.set_ylabel("change of the invsqrt cost, %")
headline(fig, "Going from 39 to 38 qubits costs about ten times what the qubit is worth",
         "One qubit of 39 pays for about 42 Toffoli-equivalents; removing the adders' second carry needs 449.")
plt.show()
print(f"Toffoli-eq per shot {TEQ:.3f}; one qubit is worth about {BREAK_EVEN:.1f} Toffoli-eq; needed {NEED}")

<a id="e4"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15.5px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #c9d6ea; border-radius: 10px; padding: 14px 18px; margin-top: 10px;">
<div style="font-size: 19px; font-weight: 700; color: #0b1f4b; margin-bottom: 6px; background: #ffffff;">4. DQI: no wire is free on the 890-qubit line</div>
<table style="border-collapse: collapse; background: #ffffff; color: #111111; font-size: 15.5px;"><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Idea</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">Qubits are a direct factor of the cost. If some wires are idle for long stretches, another value could live there and the qubit count would drop.</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Test</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">A liveness census of the whole 890-qubit file: for every step, how many wires hold a value that is used later.</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Result</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">The peak of <b>890 live wires holds for 30.5 million of 56.9 million steps</b> (54% of the run), and every wire is live for about <b>91.7%</b> of the run.</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Why it failed</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">There is no quiet stretch to borrow from: the file is at its peak for more than half of its length, so re-assigning wires cannot lower the count.</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">What it tells us</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">Fewer qubits on this line need a different algorithm layout, not wire compaction. (ThemBones went the other way: 11 more qubits for 21,740 fewer Toffolis, see Part 7.)</td></tr></table>
</div>

In [ ]:
# The DQI qubit census in one picture.
PEAK_STEPS, ALL_STEPS, LIVE = 30.5e6, 56.9e6, 0.917
fig, ax = plt.subplots(2, 1, figsize=(7, 6.4))
ax[0].barh([0], [PEAK_STEPS / 1e6], color=C[1], height=0.5); ax[0].barh([0], [(ALL_STEPS - PEAK_STEPS) / 1e6], left=[PEAK_STEPS / 1e6], color="#cfcfcf", height=0.5)
ax[0].text(PEAK_STEPS / 2e6, 0, f"at the peak of 890\n{PEAK_STEPS / 1e6:.1f}M steps ({100 * PEAK_STEPS / ALL_STEPS:.0f}%)", ha="center", va="center", color=INK, fontsize=11, fontweight="bold")
ax[0].text((PEAK_STEPS + ALL_STEPS) / 2e6, 0, "below the peak", ha="center", va="center", color=INK, fontsize=11)
ax[0].set_yticks([]); ax[0].set_xlabel("steps of the run, millions"); ax[0].set_title("How long the file sits at its peak", loc="left"); ax[0].set_xlim(0, ALL_STEPS / 1e6)
ax[1].barh([0], [100 * LIVE], color=C[0], height=0.5); ax[1].barh([0], [100 * (1 - LIVE)], left=[100 * LIVE], color="#cfcfcf", height=0.5)
ax[1].text(50 * LIVE, 0, f"live {100 * LIVE:.1f}% of the run", ha="center", va="center", color="#ffffff", fontsize=11.5, fontweight="bold")
ax[1].set_yticks([]); ax[1].set_xlim(0, 100); ax[1].set_xlabel("share of the run, %"); ax[1].set_title("A typical wire", loc="left")
headline(fig, "The 890-qubit DQI line has no idle wire to reuse")
plt.show()

<a id="e5"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15.5px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #c9d6ea; border-radius: 10px; padding: 14px 18px; margin-top: 10px;">
<div style="font-size: 19px; font-weight: 700; color: #0b1f4b; margin-bottom: 6px; background: #ffffff;">5. Chemistry: no Toffoli of the 15 loaders can be removed</div>
<table style="border-collapse: collapse; background: #ffffff; color: #111111; font-size: 15.5px;"><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Idea</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">The chemistry cost is lambda_eff &times; Toffolis &times; qubits. The 15 table loaders have a tiny input domain (1,219 addresses), so every claim about them can be checked exhaustively in seconds.</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Test</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">Our own simulation of all 15 loaders over <b>all 1,219 addresses, forward and reverse</b>. It reproduces every loader's table. Then the same exact scans as for invsqrt.</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Result</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;"><b>None of the 3,467 Toffolis per pass is removable:</b> no constant or implied control, no affine product, no repeated product, all products linearly independent, no shared-control merge, and no free wire (the live peak equals the declared qubit count in every loader).</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">Why it failed</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">The error budget leaves no room for an approximate loader either. The table has about <b>1e-7 Ha</b> of slack; flipping one least-significant bit adds <b>77 &micro;Ha</b>; deleting the Toffoli with the smallest support gives an error of <b>4.2 to 5.6 Ha against a limit of 0.020</b>.</td></tr><tr><td style="background-color: #ffffff; padding: 5px 12px 5px 0; font-weight: 700; color: #0b1f4b; vertical-align: top; white-space: nowrap;">What it tells us</td><td style="background-color: #ffffff; padding: 5px 0; color: #111111;">A loader Toffoli is not a rounding detail: each one carries table content. Savings here must come from the table and the circuit together, not from deleting gates.</td></tr></table>
</div>

In [ ]:
# Chemistry error budget on a log scale: how far each change is from the allowed error.
names = ["slack left in the table", "one LSB flip", "allowed error", "delete one Toffoli (best case)", "delete one Toffoli (worst case)"]
vals = [1e-7, 77e-6, 0.020, 4.2, 5.6]
cols = [C[2], C[3], INK2, C[7], C[7]]
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.barh(range(len(vals))[::-1], vals, color=cols, height=0.6); ax.set_xscale("log"); ax.set_xlim(1e-8, 300)
for i, v in zip(range(len(vals))[::-1], vals): ax.text(v * 1.4, i, f"{v:g} Ha", va="center", fontsize=11, color=INK)
ax.set_yticks(range(len(vals))[::-1]); ax.set_yticklabels(names); ax.set_xlabel("energy error, Hartree (log scale)")
headline(fig, "Deleting one loader Toffoli misses the error limit by a factor of 200 or more",
         "Limit 0.020 Ha. The table has 1e-7 Ha of slack, and one flipped bit already costs 77 micro-Hartree.")
plt.show()
print(f"deleting one Toffoli: {4.2 / 0.020:.0f}x to {5.6 / 0.020:.0f}x the limit; one LSB flip = {77e-6 / 1e-7:.0f}x the slack")

<a id="e6"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15.5px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #c9d6ea; border-radius: 10px; padding: 14px 18px; margin-top: 10px;">
<div style="font-size: 19px; font-weight: 700; color: #0b1f4b; margin-bottom: 6px; background: #ffffff;">6. Prices: what a Toffoli, a qubit and a failed shot are worth</div>
The board score is the geometric mean of three ratios (reference cost / your cost), so a relative saving <i>r</i> in one task lifts the score by a factor (1/(1-<i>r</i>))<sup>1/3</sup>.
<ul style="margin: 6px 0 0 0; padding-left: 22px; color: #111111;">
<li><b>invsqrt:</b> cost = qubits &times; Toffoli-equivalents per shot (rotations are priced in Toffoli-equivalents through the error they leave).</li>
<li><b>DQI:</b> cost = qubits &times; mean Toffolis per shot &times; repetition factor. The factor grows with every failed shot of the 102,400.</li>
<li><b>chemistry:</b> cost = lambda_eff &times; Toffolis &times; qubits.</li>
</ul>
<div style="margin-top: 8px;">At the 31.68 level (next cell): one DQI Toffoli = +0.000031 points, one DQI qubit = +0.0234, one invsqrt Toffoli-equivalent = +0.0065, one invsqrt qubit = +0.27. <b>Two facts that save submissions:</b> the DQI board draw depends on the file's sha256, so the same gates with a different label give a different draw; and one failed shot costs about 0.39% of the DQI cost, which is more than 1,000 Toffolis.</div>
</div>

In [ ]:
LEVEL = 31.68159
PRICES = pd.DataFrame([("DQI", "1 Toffoli per shot", 0.000031), ("DQI", "1 qubit", 0.0234), ("invsqrt", "1 Toffoli-equivalent", 0.0065), ("invsqrt", "1 qubit", 0.27)],
                      columns=["task", "saving", "board points"])
FAIL = 0.27 / 100                                  # cost of one failed DQI shot, as a fraction of the DQI cost
loss = LEVEL * (1 - (1 / (1 + FAIL)) ** (1 / 3))   # board points lost by one failed shot
PRICES.loc[len(PRICES)] = ("DQI", "1 failed shot (a loss)", -loss)
PRICES["equal to DQI Toffolis"] = (PRICES["board points"] / 0.000031).round(0).astype(int)
fig, ax = plt.subplots(figsize=(7, 4.2))
v = PRICES["board points"].abs(); yy = np.arange(len(v))[::-1]
ax.barh(yy, v, color=[C[7] if p < 0 else C[0] for p in PRICES["board points"]], height=0.6); ax.set_xscale("log"); ax.set_xlim(1e-5, 3)
for yi, p in zip(yy, PRICES["board points"]): ax.text(abs(p) * 1.3, yi, f"{p:+.6f}" if abs(p) < 0.001 else f"{p:+.4f}", va="center", fontsize=11, color=INK)
ax.set_yticks(yy); ax.set_yticklabels(PRICES["task"] + ": " + PRICES["saving"]); ax.set_xlabel("board points at the 31.68 level (log scale; red = a loss)")
headline(fig, "One failed DQI shot wipes out the gain of about 900 saved Toffolis", "Price an idea before building it: qubits are worth far more than gates, and correctness more than both.")
plt.show()
print(PRICES.to_string(index=False))

<a id="e7"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15.5px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #c9d6ea; border-radius: 10px; padding: 14px 18px; margin-top: 10px;">
<div style="font-size: 19px; font-weight: 700; color: #0b1f4b; margin-bottom: 6px; background: #ffffff;">7. The tricks of the ThemBones release, in short</div>
Summarised in our words from the method section of team ThemBones' public notebook (Danish Ali; Apache-2.0), plus almondlabs' chemistry improvement. Read their notebooks for the real description: this table is only a map.
<table style="border-collapse: collapse; background: #ffffff; color: #111111; font-size: 14.5px; margin-top: 8px; border: 1px solid #c9d6ea;">
<tr style="background: #e8eef8;"><th style="background-color: #e8eef8; padding: 6px 10px; text-align: left; color: #0b1f4b;">Task</th><th style="background-color: #e8eef8; padding: 6px 10px; text-align: left; color: #0b1f4b;">Size</th><th style="background-color: #e8eef8; padding: 6px 10px; text-align: left; color: #0b1f4b;">What they did</th></tr>
<tr><td style="background-color: #ffffff; background-color: #ffffff; padding: 6px 10px; border-top: 1px solid #d9d9d9; color: #111111; vertical-align: top;"><b>invsqrt</b></td><td style="background-color: #ffffff; background-color: #ffffff; padding: 6px 10px; border-top: 1px solid #d9d9d9; color: #111111; vertical-align: top;">39 qubits, cost 63,055.04</td><td style="background-color: #ffffff; background-color: #ffffff; padding: 6px 10px; border-top: 1px solid #d9d9d9; color: #111111; vertical-align: top;">Undo steps paid by measurement, with the phase repaid by classically controlled CZ / CCZ; rotation angles from a linear program; an exact affine clean-up pass.</td></tr>
<tr><td style="background-color: #ffffff; background-color: #ffffff; padding: 6px 10px; border-top: 1px solid #d9d9d9; color: #111111; vertical-align: top;"><b>DQI</b></td><td style="background-color: #ffffff; background-color: #ffffff; padding: 6px 10px; border-top: 1px solid #d9d9d9; color: #111111; vertical-align: top;">901 qubits (+11), 21,740 fewer Toffolis</td><td style="background-color: #ffffff; background-color: #ffffff; padding: 6px 10px; border-top: 1px solid #d9d9d9; color: #111111; vertical-align: top;">Cached 12-bit field inverses; one field product shared by two terms; direct-product kernels lifted over GF(8); measured quotient heads; exact wire re-assignment through intervals proved to be zero.</td></tr>
<tr><td style="background-color: #ffffff; background-color: #ffffff; padding: 6px 10px; border-top: 1px solid #d9d9d9; color: #111111; vertical-align: top;"><b>DQI "tickets"</b></td><td style="background-color: #ffffff; background-color: #ffffff; padding: 6px 10px; border-top: 1px solid #d9d9d9; color: #111111; vertical-align: top;">same gates</td><td style="background-color: #ffffff; background-color: #ffffff; padding: 6px 10px; border-top: 1px solid #d9d9d9; color: #111111; vertical-align: top;">Files with the same gates and different re-roll labels: the board draw depends on the file's sha256, so each label is a new draw.</td></tr>
<tr><td style="background-color: #ffffff; background-color: #ffffff; padding: 6px 10px; border-top: 1px solid #d9d9d9; color: #111111; vertical-align: top;"><b>chemistry</b></td><td style="background-color: #ffffff; background-color: #ffffff; padding: 6px 10px; border-top: 1px solid #d9d9d9; color: #111111; vertical-align: top;">405 qubits</td><td style="background-color: #ffffff; background-color: #ffffff; padding: 6px 10px; border-top: 1px solid #d9d9d9; color: #111111; vertical-align: top;">Their own table; 10 of the 15 loaders "wired"; 13 bit relations used to save gates.</td></tr>
<tr><td style="background-color: #ffffff; background-color: #ffffff; padding: 6px 10px; border-top: 1px solid #d9d9d9; color: #111111; vertical-align: top;"><b>chemistry (almondlabs)</b></td><td style="background-color: #ffffff; background-color: #ffffff; padding: 6px 10px; border-top: 1px solid #d9d9d9; color: #111111; vertical-align: top;">+0.0024 points</td><td style="background-color: #ffffff; background-color: #ffffff; padding: 6px 10px; border-top: 1px solid #d9d9d9; color: #111111; vertical-align: top;">A gauge of the O(46) channel and a Hellmann-Feynman augmented-Lagrangian L-BFGS refit (lambda_eff 46.932071 &rarr; 46.928613); loaders 5, 6 and 7 recompiled (13,436 &rarr; 13,434 T).</td></tr>
</table>
</div>

<a id="repro"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15.5px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #c9d6ea; border-radius: 10px; padding: 14px 18px; margin-top: 10px;">
<div style="font-size: 19px; font-weight: 700; color: #0b1f4b; margin-bottom: 6px; background: #ffffff;">How to reproduce, and what we did not test</div>
<b>The checks, in the order that would have saved us time:</b>
<ol style="margin: 4px 0 8px 0; padding-left: 22px; color: #111111;">
<li>Price the idea first (Part 6). A saving below one failed shot (about 900 DQI Toffolis) must be proved, not sampled.</li>
<li>Make the smallest edit that tests the claim (our void2: two gates only) before building on it.</li>
<li>For DQI run the unchanged official verifier with at least two salts; the base file must show 0 value errors in the same setup.</li>
<li>For invsqrt simulate all 3,133,452 inputs; for a chemistry loader all 1,219 addresses, forward and reverse. Both domains are small enough for an exact answer.</li>
<li>For "is this gate removable" test, on the full domain: constant or implied control, affine product, repeated product or same-target merge, measurement uncompute, zero-operand CSWAP.</li>
</ol>
<b>Not tested (so not claimed):</b> a redesign of the invsqrt for 38 qubits; XOR-merging rotations 0 and 12 of the invsqrt (only rotation 17 was tried); the T-dagger at operation 9446; a new layout of the DQI line with fewer wires; the reason at gate level why the two "void" DQI gates fire on verifier shots. The DQI results are verifier runs of single files, not proofs about all files; the invsqrt and loader results are exhaustive only for the rewrite families listed. The scans of experiments 1 and 4 were made on the 890-qubit line, those of experiment 2 on the 901-qubit file.
</div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.6;color:#111111;background:#eef3fb;border-left:6px solid #14213d;border-radius:8px;padding:12px 18px;margin:10px 0;"><div style="font-size:19px;font-weight:700;color:#14213d;margin-bottom:6px;">Credits and licence</div><a id="credits"></a>Every circuit, number and idea in this notebook comes from other people's public work; only the negative results of Part 8, the arithmetic and the explanation are ours.<ul style="margin:6px 0;padding-left:22px;color:#111111;"><li><b>Danish Ali / team ThemBones</b>: the 31.68 circuits and the method notes (<a href="https://www.kaggle.com/code/danishlynxthembones/gift-from-thembones-vqcc-31-67924" style="color:#0b3d91;background:transparent;">Gift from ThemBones</a>).</li><li><b>Almond Labs</b>: LEND29, livescan + LDH, loader recompilation, refit and gauge polish (<a href="https://www.kaggle.com/code/almondlabs/almond-labs-qcc-31-69054-demonstrable-advantage" style="color:#0b3d91;background:transparent;">31.69054 notebook</a>).</li><li><b>almondlabs</b>, <b>prvsiyan</b> (<a href="https://www.kaggle.com/code/prvsiyan/verifier-guided-quantum-circuit-optimization" style="color:#0b3d91;background:transparent;">Verifier-Guided Quantum Circuit Optimization</a>): chemistry gauge and refit.</li><li><b>maoxin123</b> (<a href="https://www.kaggle.com/code/maoxin123/qcc-33-135-3-375-toffoli-chemistry-loaders" style="color:#0b3d91;background:transparent;">3,375-Toffoli chemistry loaders, 33.135</a>): the 33.135 set (chemistry loaders and weights), on the 33.10 files below.</li><li><b>goodpjw2008</b> (<a href="https://www.kaggle.com/code/goodpjw2008/qcc-dead-word-loader-chain-dp-lb-33-10186" style="color:#0b3d91;background:transparent;">dead-word loader chain, 33.10186</a>), with the files of <b>rupam2020</b> (<a href="https://www.kaggle.com/code/rupam2020/no-bodies-vqcc-verified-portfolio" style="color:#0b3d91;background:transparent;">portfolio</a>), <b>boristown</b> (<a href="https://www.kaggle.com/code/boristown/vqcc-33-102-lb-dag-splicing-visual-guide" style="color:#0b3d91;background:transparent;">DAG splicing</a>) and <b>ozertuu</b> (<a href="https://www.kaggle.com/code/ozertuu/vqcc-milestone-public-score-31-69497" style="color:#0b3d91;background:transparent;">chemistry table</a>): the 33.10 set.</li><li><b>Rithvik Redrouthu</b> (<a href="https://www.kaggle.com/code/rithvikredrouthu/shut-up-and-calculate-vqcc-32-86114" style="color:#0b3d91;background:transparent;">37-qubit invsqrt and chemistry set</a>), <b>tonylica</b> (<a href="https://www.kaggle.com/code/tonylica/quantum-ai-32-85306-ready" style="color:#0b3d91;background:transparent;">DQI and invsqrt files</a>), <b>Areeb Ali</b> (<a href="https://www.kaggle.com/code/areebbb/qai-same-gates-better-chemistry-32-87767" style="color:#0b3d91;background:transparent;">chemistry fit</a>): the 32 to 33 files.</li><li><b>Atakan Aldemir</b>, <b>maoxin123</b>, <b>grantprater</b>, <b>alex chilton</b>, <b>somejoint</b>, <b>Koushik Rudra</b>, <b>Ertugrul Ozer</b>: the 24 to 28 steps of the staircase, as named in their notebooks.</li><li>The base lines those files build on: <b>Syed Asad Ali, Ali Haydar Ozdag, Hayato Hikawa, manwithacat, goodpjw2008, Balram Srinivasan, Lera / marculera, Manvender Singh, boristown, jeffemerson</b>.</li><li>The competition hosts at Google Quantum AI for the verifiers and scoring documents.</li></ul>Licence: Apache-2.0 for the circuits and this notebook. If a name or role is wrong or missing, say so in the comments and it will be corrected.</div>

<a id="submit"></a><div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.6;color:#111111;background:#ffffff;border:1px solid #c9d6ea;border-radius:10px;padding:12px 16px;margin-top:10px;"><div style="font-size:19px;font-weight:700;color:#0b1f4b;margin-bottom:6px;background:#ffffff;">Optional: the best public file set, checked and packaged</div>The cell below finds the 18-file set of goodpjw2008's public notebook <i>QCC | Shorter Chemistry Loaders | LB 33.21011</i> (33.21011 on the board; the same circuits are in <i>QAI | The 26-Toffoli Cut</i> by Areeb Ali and <i>33.210 Shorter Chem Loaders</i> by Mizero Lucky Gall, with another table: invsqrt with three fewer T gates built on the file of rithvikredrouthu by varianceofx and mizeroluckygall, a DQI file with zero-test cuts by goodpjw2008 on tonylica's decoder, chemistry loaders re-solved by goodpjw2008 and maoxin123 and a table refitted by goodpjw2008; all credited above and in those notebooks), checks every file by SHA-256 and writes <code>submission.zip</code> only when <code>WRITE_SUBMISSION = True</code>. It is <code>False</code> in the public version, so no submission file is left in the output. We did not re-derive these circuits.</div>

In [ ]:
import zipfile, hashlib, pathlib, shutil
WRITE_SUBMISSION = False   # False in the public version: nothing is written, no submission file is left in the output
# The best public file set (goodpjw2008's notebook output, 33.21011): the hashes below pin it. Not attached in the public version, so this cell then only prints a note.
WANT = {'invsqrt.kmb': 'bb676b95', 'dqi.kmb': '6f78ecdd', 'dfthcsos/angle_k_0.kmb': 'f2004c1f', 'dfthcsos/angle_k_1.kmb': '55d06dd8', 'dfthcsos/angle_k_2.kmb': '7524f701', 'dfthcsos/angle_k_3.kmb': '0e76c2ca', 'dfthcsos/angle_k_4.kmb': 'f2a51bbe', 'dfthcsos/angle_k_5.kmb': '6596b8f2', 'dfthcsos/angle_k_6.kmb': '7eb8bef5', 'dfthcsos/angle_k_7.kmb': 'f76cbba9', 'dfthcsos/angle_k_8.kmb': '5107abb4', 'dfthcsos/angle_k_9.kmb': '6dee8e7b', 'dfthcsos/angle_k_10.kmb': '042263af', 'dfthcsos/angle_k_11.kmb': '6d8ca068', 'dfthcsos/angle_k_12.kmb': 'c15a8f31', 'dfthcsos/angle_k_13.kmb': '7825bbba', 'dfthcsos/angle_k_14.kmb': 'adfd935a', 'dfthcsos/submission.npz': '573d10be'}
def hz(z, n):
    h = hashlib.sha256()
    with z.open(n) as f:
        for b in iter(lambda: f.read(1 << 23), b''): h.update(b)
    return h.hexdigest()
src = None
for p in sorted(pathlib.Path('/kaggle/input').rglob('submission*.zip')) if pathlib.Path('/kaggle/input').is_dir() else []:
    try:
        z = zipfile.ZipFile(p)
        if len(z.namelist()) == 18 and all(hz(z, n).startswith(h) for n, h in WANT.items()): src = (p, z); break
    except Exception as e: print('skip', p, e)
if src is None: print('source file set not attached: nothing to check (this is the public version)')
else:
    p, z = src; print('source:', p.parent.name, '- 18 members, hashes checked')
    if WRITE_SUBMISSION:
        with zipfile.ZipFile('/kaggle/working/submission.zip', 'w', zipfile.ZIP_DEFLATED) as out:
            for n in z.namelist():
                with z.open(n) as fi, out.open(zipfile.ZipInfo(n), 'w', force_zip64=True) as fo: shutil.copyfileobj(fi, fo, 1 << 23)
        print('written /kaggle/working/submission.zip')


In [ ]:
print(f"notebook finished in {time.time() - T0:.1f} s")